In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd

ROOT = Path(r"D:\2026\final_project\HF_Lung_V1_raw\original_archive")

TRAIN_DIR = ROOT / "train"
TEST_DIR = ROOT / "test"

print("Train:", TRAIN_DIR)
print("Test :", TEST_DIR)

Train: D:\2026\final_project\HF_Lung_V1_raw\original_archive\train
Test : D:\2026\final_project\HF_Lung_V1_raw\original_archive\test


In [2]:
def count_labels(folder):
    counter = Counter()
    txt_files = list(folder.glob("*_label.txt"))

    for txt_path in txt_files:
        with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                parts = line.split()

                if len(parts) >= 3:
                    label = parts[0]
                    counter[label] += 1

    return counter, len(txt_files)


train_labels, train_txt_count = count_labels(TRAIN_DIR)
test_labels, test_txt_count = count_labels(TEST_DIR)

print("Train TXT:", train_txt_count)
print(train_labels)

print()
print("Test TXT:", test_txt_count)
print(test_labels)

Train TXT: 7809
Counter({'I': 27223, 'E': 15601, 'D': 13794, 'Wheeze': 7027, 'Rhonchi': 3780, 'Stridor': 657})

Test TXT: 1956
Counter({'I': 6872, 'E': 2748, 'D': 1812, 'Wheeze': 1430, 'Rhonchi': 960, 'Stridor': 29})


In [3]:
all_labels = sorted(set(train_labels) | set(test_labels))

label_df = pd.DataFrame({
    "Label": all_labels,
    "Train_Count": [train_labels.get(x, 0) for x in all_labels],
    "Test_Count": [test_labels.get(x, 0) for x in all_labels],
})

label_df["Total"] = (
    label_df["Train_Count"] +
    label_df["Test_Count"]
)

label_df

,Label,Train_Count,Test_Count,Total
0,D,13794,1812,15606
1,E,15601,2748,18349
2,I,27223,6872,34095
3,Rhonchi,3780,960,4740
4,Stridor,657,29,686
5,Wheeze,7027,1430,8457


In [4]:
from pathlib import Path
import pandas as pd

sample_txt = TRAIN_DIR / "steth_20180814_09_37_11_label.txt"

rows = []

with open(sample_txt, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        parts = line.strip().split()

        if len(parts) >= 3:
            rows.append({
                "label": parts[0],
                "start": parts[1],
                "end": parts[2],
            })

sample_df = pd.DataFrame(rows)
sample_df

,label,start,end
0,I,00:00:01.500,00:00:02.457
1,D,00:00:01.500,00:00:02.457
2,E,00:00:02.608,00:00:03.637
3,D,00:00:02.608,00:00:03.637
4,I,00:00:05.227,00:00:06.350
5,D,00:00:05.227,00:00:06.350
6,E,00:00:06.431,00:00:07.411
7,D,00:00:06.431,00:00:07.411
8,I,00:00:09.038,00:00:10.019
9,E,00:00:10.205,00:00:11.186


In [5]:
from pathlib import Path
from collections import Counter
import pandas as pd


def time_to_sec(t):
    # 예: 00:00:12.754
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)


def has_overlap(a_start, a_end, b_start, b_end):
    # 겹치는 시간이 조금이라도 있으면 True
    return max(a_start, b_start) < min(a_end, b_end)


def classify_segments(folder):
    results = []

    txt_files = sorted(folder.glob("*_label.txt"))

    for txt_path in txt_files:
        events = []

        with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) >= 3:
                    label = parts[0]
                    start = time_to_sec(parts[1])
                    end = time_to_sec(parts[2])

                    events.append({
                        "label": label,
                        "start": start,
                        "end": end
                    })

        # 호흡 phase 후보: I / E
        phases = [x for x in events if x["label"] in ["I", "E"]]

        for phase in phases:
            p_start = phase["start"]
            p_end = phase["end"]

            overlapping_labels = []

            for event in events:
                if event["label"] in ["I", "E"]:
                    continue

                if has_overlap(
                    p_start, p_end,
                    event["start"], event["end"]
                ):
                    overlapping_labels.append(event["label"])

            overlap_set = set(overlapping_labels)

            has_d = "D" in overlap_set
            has_wheeze = "Wheeze" in overlap_set
            has_rhonchi = "Rhonchi" in overlap_set
            has_stridor = "Stridor" in overlap_set

            # Rhonchi / Stridor가 있으면 4-class 외부검증에서는 제외
            if has_rhonchi or has_stridor:
                final_class = "EXCLUDE"
            else:
                if has_d and has_wheeze:
                    final_class = "Both"
                elif has_d:
                    final_class = "Crackle"
                elif has_wheeze:
                    final_class = "Wheeze"
                else:
                    final_class = "Normal"

            results.append({
                "file": txt_path.name,
                "phase": phase["label"],
                "start_sec": p_start,
                "end_sec": p_end,
                "duration_sec": p_end - p_start,
                "has_D": has_d,
                "has_Wheeze": has_wheeze,
                "has_Rhonchi": has_rhonchi,
                "has_Stridor": has_stridor,
                "class": final_class
            })

    return pd.DataFrame(results)

In [6]:
train_segments = classify_segments(TRAIN_DIR)
test_segments = classify_segments(TEST_DIR)

print("Train segments:", len(train_segments))
print("Test segments :", len(test_segments))

Train segments: 42824
Test segments : 9620


In [7]:
print("TRAIN")
display(
    train_segments["class"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

print()

print("TEST")
display(
    test_segments["class"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

TRAIN


,Class,Count
0,Normal,19647
1,Crackle,13619
2,Wheeze,5101
3,EXCLUDE,4111
4,Both,346



TEST


,Class,Count
0,Normal,5735
1,Crackle,1731
2,Wheeze,1195
3,EXCLUDE,945
4,Both,14


In [8]:
summary = pd.DataFrame({
    "Train": train_segments["class"].value_counts(),
    "Test": test_segments["class"].value_counts()
}).fillna(0).astype(int)

summary["Total"] = summary["Train"] + summary["Test"]

summary

,Train,Test,Total
class,,,
Normal,19647,5735,25382
Crackle,13619,1731,15350
Wheeze,5101,1195,6296
EXCLUDE,4111,945,5056
Both,346,14,360


In [9]:
print("Train phase counts")
print(train_segments["phase"].value_counts())

print()

print("Test phase counts")
print(test_segments["phase"].value_counts())

Train phase counts
phase
I    27223
E    15601
Name: count, dtype: int64

Test phase counts
phase
I    6872
E    2748
Name: count, dtype: int64


In [10]:
def analyze_phase_transitions(folder):
    rows = []

    txt_files = sorted(folder.glob("*_label.txt"))

    for txt_path in txt_files:
        phases = []

        with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) >= 3 and parts[0] in ["I", "E"]:
                    phases.append({
                        "label": parts[0],
                        "start": time_to_sec(parts[1]),
                        "end": time_to_sec(parts[2])
                    })

        phases = sorted(phases, key=lambda x: x["start"])

        for i in range(len(phases) - 1):
            current = phases[i]
            next_phase = phases[i + 1]

            rows.append({
                "file": txt_path.name,
                "current": current["label"],
                "next": next_phase["label"],
                "transition": f'{current["label"]}->{next_phase["label"]}',
                "current_start": current["start"],
                "current_end": current["end"],
                "next_start": next_phase["start"],
                "next_end": next_phase["end"],
                "gap_sec": next_phase["start"] - current["end"]
            })

    return pd.DataFrame(rows)


train_transition = analyze_phase_transitions(TRAIN_DIR)
test_transition = analyze_phase_transitions(TEST_DIR)

In [11]:
print("TRAIN transition")
display(
    train_transition["transition"]
    .value_counts()
    .rename_axis("Transition")
    .reset_index(name="Count")
)

print("TEST transition")
display(
    test_transition["transition"]
    .value_counts()
    .rename_axis("Transition")
    .reset_index(name="Count")
)

TRAIN transition


,Transition,Count
0,I->E,14701
1,E->I,12123
2,I->I,8112
3,E->E,329


TEST transition


,Transition,Count
0,I->I,2955
1,I->E,2602
2,E->I,2095
3,E->E,28


In [12]:
train_ie = train_transition[
    train_transition["transition"] == "I->E"
]

test_ie = test_transition[
    test_transition["transition"] == "I->E"
]

print("TRAIN I->E gap")
display(train_ie["gap_sec"].describe())

print("TEST I->E gap")
display(test_ie["gap_sec"].describe())

TRAIN I->E gap


count    14701.000000
mean         0.136037
std          0.242414
min         -1.349000
25%          0.007000
50%          0.052000
75%          0.178000
max          4.459000
Name: gap_sec, dtype: float64

TEST I->E gap


count    2602.000000
mean        0.233080
std         0.351439
min        -1.099000
25%         0.022000
50%         0.113500
75%         0.329000
max         4.401000
Name: gap_sec, dtype: float64

In [13]:
def gap_summary(df):
    return pd.Series({
        "gap < 0 sec": (df["gap_sec"] < 0).sum(),
        "0 ~ 0.5 sec": ((df["gap_sec"] >= 0) & (df["gap_sec"] <= 0.5)).sum(),
        "0.5 ~ 1.0 sec": ((df["gap_sec"] > 0.5) & (df["gap_sec"] <= 1.0)).sum(),
        "1.0 ~ 2.0 sec": ((df["gap_sec"] > 1.0) & (df["gap_sec"] <= 2.0)).sum(),
        "> 2.0 sec": (df["gap_sec"] > 2.0).sum()
    })

print("TRAIN I->E")
display(gap_summary(train_ie).to_frame("Count"))

print("TEST I->E")
display(gap_summary(test_ie).to_frame("Count"))

TRAIN I->E


,Count
gap < 0 sec,1788
0 ~ 0.5 sec,12101
0.5 ~ 1.0 sec,650
1.0 ~ 2.0 sec,126
> 2.0 sec,36


TEST I->E


,Count
gap < 0 sec,184
0 ~ 0.5 sec,2078
0.5 ~ 1.0 sec,266
1.0 ~ 2.0 sec,58
> 2.0 sec,16


In [14]:
def build_ie_cycle_candidates(folder):
    rows = []

    txt_files = sorted(folder.glob("*_label.txt"))

    for txt_path in txt_files:
        events = []

        with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) >= 3:
                    events.append({
                        "label": parts[0],
                        "start": time_to_sec(parts[1]),
                        "end": time_to_sec(parts[2])
                    })

        phases = sorted(
            [x for x in events if x["label"] in ["I", "E"]],
            key=lambda x: x["start"]
        )

        # 실제로 서로 인접한 I -> E만 후보
        for i in range(len(phases) - 1):
            first = phases[i]
            second = phases[i + 1]

            if first["label"] != "I" or second["label"] != "E":
                continue

            cycle_start = first["start"]
            cycle_end = second["end"]

            gap_sec = second["start"] - first["end"]
            cycle_duration = cycle_end - cycle_start

            overlapping_labels = []

            for event in events:
                if event["label"] in ["I", "E"]:
                    continue

                if has_overlap(
                    cycle_start,
                    cycle_end,
                    event["start"],
                    event["end"]
                ):
                    overlapping_labels.append(event["label"])

            overlap_set = set(overlapping_labels)

            has_d = "D" in overlap_set
            has_wheeze = "Wheeze" in overlap_set
            has_rhonchi = "Rhonchi" in overlap_set
            has_stridor = "Stridor" in overlap_set

            if has_rhonchi or has_stridor:
                final_class = "EXCLUDE"
            elif has_d and has_wheeze:
                final_class = "Both"
            elif has_d:
                final_class = "Crackle"
            elif has_wheeze:
                final_class = "Wheeze"
            else:
                final_class = "Normal"

            rows.append({
                "file": txt_path.name,
                "cycle_start": cycle_start,
                "cycle_end": cycle_end,
                "gap_sec": gap_sec,
                "duration_sec": cycle_duration,
                "has_D": has_d,
                "has_Wheeze": has_wheeze,
                "has_Rhonchi": has_rhonchi,
                "has_Stridor": has_stridor,
                "class": final_class
            })

    return pd.DataFrame(rows)

In [15]:
train_cycles = build_ie_cycle_candidates(TRAIN_DIR)
test_cycles = build_ie_cycle_candidates(TEST_DIR)

print("Train I->E cycle candidates:", len(train_cycles))
print("Test I->E cycle candidates :", len(test_cycles))

Train I->E cycle candidates: 14701
Test I->E cycle candidates : 2602


In [16]:
print("TRAIN cycle duration")
display(train_cycles["duration_sec"].describe())

print("TEST cycle duration")
display(test_cycles["duration_sec"].describe())

TRAIN cycle duration


count    14701.000000
mean         1.966731
std          0.686173
min          0.454000
25%          1.520000
50%          1.815000
75%          2.241000
max          7.459000
Name: duration_sec, dtype: float64

TEST cycle duration


count    2602.000000
mean        2.097724
std         0.679045
min         0.865000
25%         1.729000
50%         1.945000
75%         2.241000
max        10.527000
Name: duration_sec, dtype: float64

In [17]:
def duration_summary(df):
    return pd.Series({
        "<= 2.7 sec": (df["duration_sec"] <= 2.7).sum(),
        "<= 3.0 sec": (df["duration_sec"] <= 3.0).sum(),
        "<= 4.0 sec": (df["duration_sec"] <= 4.0).sum(),
        "> 4.0 sec":  (df["duration_sec"] > 4.0).sum()
    })

print("TRAIN")
display(duration_summary(train_cycles).to_frame("Count"))

print("TEST")
display(duration_summary(test_cycles).to_frame("Count"))

TRAIN


,Count
<= 2.7 sec,12893
<= 3.0 sec,13510
<= 4.0 sec,14436
> 4.0 sec,265


TEST


,Count
<= 2.7 sec,2276
<= 3.0 sec,2423
<= 4.0 sec,2551
> 4.0 sec,51


In [18]:
print("TRAIN cycle classes")
display(
    train_cycles["class"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

print("TEST cycle classes")
display(
    test_cycles["class"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

TRAIN cycle classes


,Class,Count
0,Crackle,5354
1,Normal,4702
2,EXCLUDE,2183
3,Wheeze,1598
4,Both,864


TEST cycle classes


,Class,Count
0,Normal,1397
1,EXCLUDE,497
2,Crackle,482
3,Wheeze,185
4,Both,41


In [19]:
def make_external_subset(df):
    return df[
        (df["duration_sec"] <= 4.0) &
        (df["class"] != "EXCLUDE")
    ].copy()


train_external = make_external_subset(train_cycles)
test_external = make_external_subset(test_cycles)

print("TRAIN external candidates:", len(train_external))
print("TEST external candidates :", len(test_external))

print("\nTRAIN")
display(
    train_external["class"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

print("\nTEST")
display(
    test_external["class"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

TRAIN external candidates: 12315
TEST external candidates : 2058

TRAIN


,Class,Count
0,Crackle,5293
1,Normal,4658
2,Wheeze,1544
3,Both,820



TEST


,Class,Count
0,Normal,1358
1,Crackle,476
2,Wheeze,183
3,Both,41


In [20]:
test_class_summary = (
    test_external["class"]
    .value_counts()
    .rename_axis("Class")
    .reset_index(name="Count")
)

test_class_summary["Percent"] = (
    test_class_summary["Count"]
    / test_class_summary["Count"].sum()
    * 100
).round(2)

test_class_summary

,Class,Count,Percent
0,Normal,1358,65.99
1,Crackle,476,23.13
2,Wheeze,183,8.89
3,Both,41,1.99


In [21]:
import soundfile as sf

sample_wavs = sorted(TEST_DIR.glob("*.wav"))[:10]

audio_info = []

for wav_path in sample_wavs:
    info = sf.info(wav_path)

    audio_info.append({
        "file": wav_path.name,
        "samplerate": info.samplerate,
        "channels": info.channels,
        "duration": info.duration,
        "subtype": info.subtype
    })

pd.DataFrame(audio_info)

,file,samplerate,channels,duration,subtype
0,steth_20190801_09_46_05.wav,4000,1,15.0,PCM_16
1,steth_20190801_09_46_26.wav,4000,1,15.0,PCM_16
2,steth_20190801_09_46_51.wav,4000,1,15.0,PCM_16
3,steth_20190801_09_47_36.wav,4000,1,15.0,PCM_16
4,steth_20190801_09_47_58.wav,4000,1,15.0,PCM_16
5,steth_20190801_09_48_31.wav,4000,1,15.0,PCM_16
6,steth_20190801_09_49_03.wav,4000,1,15.0,PCM_16
7,steth_20190801_09_51_07.wav,4000,1,15.0,PCM_16
8,steth_20190801_09_52_21.wav,4000,1,15.0,PCM_16
9,steth_20190801_09_52_41.wav,4000,1,15.0,PCM_16


In [22]:
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf

OUTPUT_ROOT = Path(
    r"D:\2026\final_project\external_data_processed\HF_Lung_V1"
)

TEST_OUTPUT = OUTPUT_ROOT / "test_cycles_4k_4sec"
METADATA_DIR = OUTPUT_ROOT / "metadata"

CLASS_NAMES = ["Normal", "Crackle", "Wheeze", "Both"]

for class_name in CLASS_NAMES:
    (TEST_OUTPUT / class_name).mkdir(parents=True, exist_ok=True)

METADATA_DIR.mkdir(parents=True, exist_ok=True)

# 원래 15초 WAV 안에서 몇 번째 I->E cycle인지 기록
test_cycles = test_cycles.copy()
test_cycles["cycle_index"] = (
    test_cycles.groupby("file").cumcount() + 1
)

# 앞서 확정한 외부검증 조건
test_external = test_cycles[
    (test_cycles["duration_sec"] <= 4.0) &
    (test_cycles["class"] != "EXCLUDE")
].copy()

print("External test cycles:", len(test_external))
print()
print(test_external["class"].value_counts())

External test cycles: 2058

class
Normal     1358
Crackle     476
Wheeze      183
Both         41
Name: count, dtype: int64


In [23]:
TARGET_SR = 4000
TARGET_DURATION = 4.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_DURATION)  # 16000


def make_4sec_cycle(wav_path, start_sec, end_sec):
    audio, sr = sf.read(wav_path, dtype="float32")

    if sr != TARGET_SR:
        raise ValueError(
            f"Unexpected sample rate: {wav_path.name} / {sr} Hz"
        )

    # mono 확인
    if audio.ndim != 1:
        raise ValueError(
            f"Unexpected channels: {wav_path.name} / shape={audio.shape}"
        )

    start_sample = int(round(start_sec * sr))
    end_sample = int(round(end_sec * sr))

    segment = audio[start_sample:end_sample]

    original_samples = len(segment)

    if original_samples < TARGET_SAMPLES:
        total_pad = TARGET_SAMPLES - original_samples

        left_pad = total_pad // 2
        right_pad = total_pad - left_pad

        segment = np.pad(
            segment,
            (left_pad, right_pad),
            mode="constant"
        )

        method = "symmetric_zero_padding"

    elif original_samples > TARGET_SAMPLES:
        # 현재 subset은 <= 4초이므로 원칙적으로 발생하면 안 됨
        excess = original_samples - TARGET_SAMPLES

        left_crop = excess // 2
        segment = segment[
            left_crop:left_crop + TARGET_SAMPLES
        ]

        method = "center_crop"

    else:
        method = "none"

    return segment, original_samples, method

In [24]:
dry_run_rows = []

for _, row in test_external.head(10).iterrows():

    label_filename = row["file"]

    wav_filename = label_filename.replace(
        "_label.txt",
        ".wav"
    )

    wav_path = TEST_DIR / wav_filename

    segment, original_samples, method = make_4sec_cycle(
        wav_path,
        row["cycle_start"],
        row["cycle_end"]
    )

    output_name = (
        f"{wav_path.stem}"
        f"_cycle{int(row['cycle_index']):03d}"
        f"_{row['class']}.wav"
    )

    output_path = (
        TEST_OUTPUT /
        row["class"] /
        output_name
    )

    sf.write(
        output_path,
        segment,
        TARGET_SR,
        subtype="PCM_16"
    )

    info = sf.info(output_path)

    dry_run_rows.append({
        "source_wav": wav_filename,
        "cycle_index": int(row["cycle_index"]),
        "class": row["class"],
        "start_sec": row["cycle_start"],
        "end_sec": row["cycle_end"],
        "original_duration": row["duration_sec"],
        "original_samples": original_samples,
        "method": method,
        "output_samples": info.frames,
        "output_sr": info.samplerate,
        "output_duration": info.duration,
        "output_file": output_name
    })

dry_run_df = pd.DataFrame(dry_run_rows)

dry_run_df

,source_wav,cycle_index,class,start_sec,end_sec,original_duration,original_samples,method,output_samples,output_sr,output_duration,output_file
0,steth_20190801_09_51_07.wav,1,Normal,0.361,2.145,1.784,7136,symmetric_zero_padding,16000,4000,4.0,steth_20190801_09_51_07_cycle001_Normal.wav
1,steth_20190801_09_51_07.wav,2,Normal,2.315,3.929,1.614,6456,symmetric_zero_padding,16000,4000,4.0,steth_20190801_09_51_07_cycle002_Normal.wav
2,steth_20190801_09_51_07.wav,3,Normal,4.213,5.864,1.651,6604,symmetric_zero_padding,16000,4000,4.0,steth_20190801_09_51_07_cycle003_Normal.wav
3,steth_20190801_09_51_07.wav,4,Normal,6.167,7.762,1.595,6380,symmetric_zero_padding,16000,4000,4.0,steth_20190801_09_51_07_cycle004_Normal.wav
4,steth_20190801_09_51_07.wav,5,Normal,7.969,9.906,1.937,7748,symmetric_zero_padding,16000,4000,4.0,steth_20190801_09_51_07_cycle005_Normal.wav
5,steth_20190801_09_51_07.wav,6,Normal,10.132,11.670,1.538,6152,symmetric_zero_padding,16000,4000,4.0,steth_20190801_09_51_07_cycle006_Normal.wav
6,steth_20190801_09_51_07.wav,7,Normal,12.029,13.548,1.519,6076,symmetric_zero_padding,16000,4000,4.0,steth_20190801_09_51_07_cycle007_Normal.wav
7,steth_20190801_09_53_09.wav,1,Crackle,0.000,3.657,3.657,14628,symmetric_zero_padding,16000,4000,4.0,steth_20190801_09_53_09_cycle001_Crackle.wav
8,steth_20190801_09_53_09.wav,2,Crackle,3.698,6.875,3.177,12708,symmetric_zero_padding,16000,4000,4.0,steth_20190801_09_53_09_cycle002_Crackle.wav
9,steth_20190801_09_53_09.wav,3,Crackle,8.182,9.530,1.348,5392,symmetric_zero_padding,16000,4000,4.0,steth_20190801_09_53_09_cycle003_Crackle.wav


In [25]:
generated_rows = []

for _, row in test_external.iterrows():

    label_filename = row["file"]

    wav_filename = label_filename.replace(
        "_label.txt",
        ".wav"
    )

    wav_path = TEST_DIR / wav_filename

    segment, original_samples, method = make_4sec_cycle(
        wav_path,
        row["cycle_start"],
        row["cycle_end"]
    )

    output_name = (
        f"{wav_path.stem}"
        f"_cycle{int(row['cycle_index']):03d}"
        f"_{row['class']}.wav"
    )

    output_path = (
        TEST_OUTPUT /
        row["class"] /
        output_name
    )

    sf.write(
        output_path,
        segment,
        TARGET_SR,
        subtype="PCM_16"
    )

    info = sf.info(output_path)

    generated_rows.append({
        "source_wav": wav_filename,
        "cycle_index": int(row["cycle_index"]),
        "class": row["class"],
        "cycle_start_sec": row["cycle_start"],
        "cycle_end_sec": row["cycle_end"],
        "original_duration_sec": row["duration_sec"],
        "original_samples": original_samples,
        "normalization_method": method,
        "output_samples": info.frames,
        "output_sr": info.samplerate,
        "output_duration_sec": info.duration,
        "output_file": output_name
    })


hf_test_metadata = pd.DataFrame(generated_rows)

print("Generated:", len(hf_test_metadata))

Generated: 2058


In [26]:
for class_name in CLASS_NAMES:
    class_dir = TEST_OUTPUT / class_name
    count = len(list(class_dir.glob("*.wav")))
    print(f"{class_name}: {count}")

Normal: 1358
Crackle: 476
Wheeze: 183
Both: 41


In [27]:
print("Sample rate unique:")
print(hf_test_metadata["output_sr"].unique())

print("\nOutput samples unique:")
print(hf_test_metadata["output_samples"].unique())

print("\nOutput duration unique:")
print(hf_test_metadata["output_duration_sec"].unique())

print("\nNormalization method:")
print(hf_test_metadata["normalization_method"].value_counts())

Sample rate unique:
[4000]

Output samples unique:
[16000]

Output duration unique:
[4.]

Normalization method:
normalization_method
symmetric_zero_padding    2058
Name: count, dtype: int64


In [28]:
metadata_path = (
    METADATA_DIR /
    "HF_Lung_V1_external_test_metadata.csv"
)

hf_test_metadata.to_csv(
    metadata_path,
    index=False,
    encoding="utf-8-sig"
)

print(metadata_path)

D:\2026\final_project\external_data_processed\HF_Lung_V1\metadata\HF_Lung_V1_external_test_metadata.csv


In [29]:
import torch
import torch.nn as nn
from torchvision import models
import torchaudio
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = Path(
    r"D:\2026\final_project\results\checkpoints\AlexNet_Base_Final_Selected_seed777.pth"
)

CLASS_NAMES_MODEL = ["Both", "Crackle", "Normal", "Wheeze"]

print("Device:", DEVICE)
print("Checkpoint exists:", CHECKPOINT_PATH.exists())

Device: cpu
Checkpoint exists: True


In [30]:
model = models.alexnet(weights=None)

model.classifier[6] = nn.Linear(
    model.classifier[6].in_features,
    4
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

# checkpoint 저장 형태에 대응
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]

elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]

else:
    state_dict = checkpoint

model.load_state_dict(state_dict)

model = model.to(DEVICE)
model.eval()

print("AlexNet loaded successfully.")

AlexNet loaded successfully.


In [31]:
SR = 4000
N_FFT = 512
HOP_LENGTH = 128
N_MELS = 128
F_MIN = 0
F_MAX = 2000

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=F_MIN,
    f_max=F_MAX,
    power=2.0
)


def wav_to_alexnet_input(wav_path):
    waveform, sr = torchaudio.load(str(wav_path))

    if sr != SR:
        raise ValueError(
            f"Unexpected sample rate: {sr}"
        )

    # mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    mel = mel_transform(waveform)

    # relative dB
    mel_db = 10.0 * torch.log10(
        torch.clamp(mel, min=1e-10)
    )

    mel_db = mel_db - mel_db.max()
    mel_db = torch.clamp(mel_db, min=-80.0, max=0.0)

    # [-80, 0] → [0, 1]
    mel_norm = (mel_db + 80.0) / 80.0

    # [1, 128, 126] → [1, 1, 224, 224]
    x = mel_norm.unsqueeze(0)

    x = F.interpolate(
        x,
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    )

    # AlexNet RGB 입력 형태
    x = x.repeat(1, 3, 1, 1)

    return x

In [32]:
sample_paths = []

for class_name in ["Normal", "Crackle", "Wheeze", "Both"]:
    files = sorted(
        (TEST_OUTPUT / class_name).glob("*.wav")
    )

    sample_paths.extend(files[:2])

len(sample_paths)

8

In [33]:
smoke_results = []

with torch.inference_mode():

    for wav_path in sample_paths:

        x = wav_to_alexnet_input(wav_path).to(DEVICE)

        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

        pred_idx = int(np.argmax(probs))
        pred_class = CLASS_NAMES_MODEL[pred_idx]

        true_class = wav_path.parent.name

        smoke_results.append({
            "file": wav_path.name,
            "true_class": true_class,
            "pred_class": pred_class,
            "Both": probs[0],
            "Crackle": probs[1],
            "Normal": probs[2],
            "Wheeze": probs[3],
            "confidence": probs[pred_idx]
        })


smoke_df = pd.DataFrame(smoke_results)

smoke_df

,file,true_class,pred_class,Both,Crackle,Normal,Wheeze,confidence
0,steth_20190801_09_51_07_cycle001_Normal.wav,Normal,Normal,0.000016,0.101971,0.885618,0.012395,0.885618
1,steth_20190801_09_51_07_cycle002_Normal.wav,Normal,Normal,0.000001,0.020676,0.908726,0.070597,0.908726
2,steth_20190801_09_53_09_cycle001_Crackle.wav,Crackle,Wheeze,0.110321,0.152549,0.237558,0.499572,0.499572
3,steth_20190801_09_53_09_cycle002_Crackle.wav,Crackle,Normal,0.007037,0.161493,0.722479,0.108991,0.722479
4,steth_20190805_09_46_42_cycle003_Wheeze.wav,Wheeze,Normal,0.000014,0.126330,0.866472,0.007184,0.866472
5,steth_20190806_16_43_10_cycle001_Wheeze.wav,Wheeze,Wheeze,0.001010,0.000524,0.004985,0.993481,0.993481
6,steth_20190809_10_30_07_cycle001_Both.wav,Both,Wheeze,0.029403,0.040215,0.211696,0.718685,0.718685
7,steth_20190809_10_30_07_cycle002_Both.wav,Both,Wheeze,0.047752,0.089137,0.403985,0.459127,0.459127


In [34]:
prob_cols = ["Both", "Crackle", "Normal", "Wheeze"]

smoke_df["prob_sum"] = smoke_df[prob_cols].sum(axis=1)

smoke_df[
    ["true_class", "pred_class", "confidence", "prob_sum"]
]

,true_class,pred_class,confidence,prob_sum
0,Normal,Normal,0.885618,1.0
1,Normal,Normal,0.908726,1.0
2,Crackle,Wheeze,0.499572,1.0
3,Crackle,Normal,0.722479,1.0
4,Wheeze,Normal,0.866472,1.0
5,Wheeze,Wheeze,0.993481,1.0
6,Both,Wheeze,0.718685,1.0
7,Both,Wheeze,0.459127,1.0


In [35]:
from tqdm.auto import tqdm

external_results = []

with torch.inference_mode():

    for _, row in tqdm(
        hf_test_metadata.iterrows(),
        total=len(hf_test_metadata),
        desc="HF_Lung external validation"
    ):

        true_class = row["class"]

        wav_path = (
            TEST_OUTPUT /
            true_class /
            row["output_file"]
        )

        x = wav_to_alexnet_input(wav_path).to(DEVICE)

        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

        pred_idx = int(np.argmax(probs))
        pred_class = CLASS_NAMES_MODEL[pred_idx]

        external_results.append({
            "file": row["output_file"],
            "source_wav": row["source_wav"],
            "true_class": true_class,
            "pred_class": pred_class,

            "P_Both": float(probs[0]),
            "P_Crackle": float(probs[1]),
            "P_Normal": float(probs[2]),
            "P_Wheeze": float(probs[3]),

            "confidence": float(probs[pred_idx]),
            "correct": true_class == pred_class
        })


external_results_df = pd.DataFrame(external_results)

print("Completed:", len(external_results_df))
external_results_df.head()

C:\Users\정용호\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
HF_Lung external validation: 100%|██████████| 2058/2058 [00:31<00:00, 66.22it/s]

Completed: 2058


,file,source_wav,true_class,pred_class,P_Both,P_Crackle,P_Normal,P_Wheeze,confidence,correct
0,steth_20190801_09_51_07_cycle001_Normal.wav,steth_20190801_09_51_07.wav,Normal,Normal,0.000016,0.101971,0.885618,0.012395,0.885618,True
1,steth_20190801_09_51_07_cycle002_Normal.wav,steth_20190801_09_51_07.wav,Normal,Normal,0.000001,0.020676,0.908726,0.070597,0.908726,True
2,steth_20190801_09_51_07_cycle003_Normal.wav,steth_20190801_09_51_07.wav,Normal,Normal,0.000867,0.046088,0.560892,0.392153,0.560892,True
3,steth_20190801_09_51_07_cycle004_Normal.wav,steth_20190801_09_51_07.wav,Normal,Normal,0.000057,0.046047,0.769390,0.184506,0.769390,True
4,steth_20190801_09_51_07_cycle005_Normal.wav,steth_20190801_09_51_07.wav,Normal,Normal,0.000674,0.292290,0.652065,0.054971,0.652065,True


In [36]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score
)

y_true = external_results_df["true_class"]
y_pred = external_results_df["pred_class"]

EVAL_CLASSES = [
    "Both",
    "Crackle",
    "Normal",
    "Wheeze"
]

accuracy = accuracy_score(y_true, y_pred)

balanced_acc = balanced_accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=EVAL_CLASSES,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=EVAL_CLASSES,
    average="weighted",
    zero_division=0
)

print(f"Accuracy          : {accuracy:.6f}")
print(f"Balanced Accuracy : {balanced_acc:.6f}")
print(f"Macro F1          : {macro_f1:.6f}")
print(f"Weighted F1       : {weighted_f1:.6f}")

Accuracy          : 0.362002
Balanced Accuracy : 0.289346
Macro F1          : 0.230598
Weighted F1       : 0.401798


In [37]:
report = classification_report(
    y_true,
    y_pred,
    labels=EVAL_CLASSES,
    target_names=EVAL_CLASSES,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

report_df

,precision,recall,f1-score,support
Both,0.000000,0.000000,0.000000,41.000000
Crackle,0.171053,0.136555,0.151869,476.000000
Normal,0.690909,0.419735,0.522217,1358.000000
Wheeze,0.156472,0.601093,0.248307,183.000000
accuracy,0.362002,0.362002,0.362002,0.362002
macro avg,0.254608,0.289346,0.230598,2058.000000
weighted avg,0.509383,0.362002,0.401798,2058.000000


In [38]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=EVAL_CLASSES
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True_{x}" for x in EVAL_CLASSES],
    columns=[f"Pred_{x}" for x in EVAL_CLASSES]
)

cm_df

,Pred_Both,Pred_Crackle,Pred_Normal,Pred_Wheeze
True_Both,0,9,9,23
True_Crackle,13,65,232,166
True_Normal,93,291,570,404
True_Wheeze,44,15,14,110


In [39]:
RESULT_DIR = (
    OUTPUT_ROOT /
    "results"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

external_results_df.to_csv(
    RESULT_DIR /
    "HF_Lung_V1_external_validation_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

report_df.to_csv(
    RESULT_DIR /
    "HF_Lung_V1_external_validation_classification_report.csv",
    encoding="utf-8-sig"
)

cm_df.to_csv(
    RESULT_DIR /
    "HF_Lung_V1_external_validation_confusion_matrix.csv",
    encoding="utf-8-sig"
)

print("Saved to:")
print(RESULT_DIR)

Saved to:
D:\2026\final_project\external_data_processed\HF_Lung_V1\results


## HF_Lung_V1 External Validation Summary

- External dataset: HF_Lung_V1 official test split
- Original WAV: 4,000 Hz, mono, PCM16, 15 sec
- External validation cycles: 2,058
  - Normal: 1,358
  - Crackle: 476
  - Wheeze: 183
  - Both: 41
- Rhonchi / Stridor overlapping cycles were excluded.
- Only adjacent I→E respiratory-cycle candidates with duration ≤ 4 sec were included.
- Shorter cycles were standardized to 4 sec using symmetric zero-padding.
- The ICBHI-trained AlexNet checkpoint was used without retraining or model reselection.

### Performance
- Accuracy: 0.362002
- Balanced Accuracy: 0.289346
- Macro F1: 0.230598
- Weighted F1: 0.401798

### Interpretation
Performance decreased compared with the internal ICBHI test set.
The result suggests limited cross-dataset generalization under differences in
recording conditions, annotation structure, and acoustic distribution.

This analysis should be interpreted as exploratory external validation because
HF_Lung_V1 does not provide the same native four-class respiratory-cycle
annotation scheme as ICBHI.

In [40]:
comparison_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Macro F1",
        "Weighted F1"
    ],
    "ICBHI_Internal_Test": [
        0.527505,
        0.459836,
        0.424701,
        0.528600
    ],
    "HF_Lung_External": [
        accuracy,
        balanced_acc,
        macro_f1,
        weighted_f1
    ]
})

comparison_df["Difference"] = (
    comparison_df["HF_Lung_External"]
    - comparison_df["ICBHI_Internal_Test"]
)

comparison_df

,Metric,ICBHI_Internal_Test,HF_Lung_External,Difference
0,Accuracy,0.527505,0.362002,-0.165503
1,Balanced Accuracy,0.459836,0.289346,-0.170490
2,Macro F1,0.424701,0.230598,-0.194103
3,Weighted F1,0.528600,0.401798,-0.126802


In [41]:
recall_comparison_df = pd.DataFrame({
    "Class": [
        "Both",
        "Crackle",
        "Normal",
        "Wheeze"
    ],
    "ICBHI_Recall": [
        0.052,
        0.524,
        0.540,
        0.723
    ],
    "HF_Lung_Recall": [
        report["Both"]["recall"],
        report["Crackle"]["recall"],
        report["Normal"]["recall"],
        report["Wheeze"]["recall"]
    ]
})

recall_comparison_df["Difference"] = (
    recall_comparison_df["HF_Lung_Recall"]
    - recall_comparison_df["ICBHI_Recall"]
)

recall_comparison_df

,Class,ICBHI_Recall,HF_Lung_Recall,Difference
0,Both,0.052,0.000000,-0.052000
1,Crackle,0.524,0.136555,-0.387445
2,Normal,0.540,0.419735,-0.120265
3,Wheeze,0.723,0.601093,-0.121907


In [42]:
true_distribution = (
    external_results_df["true_class"]
    .value_counts()
    .reindex(EVAL_CLASSES, fill_value=0)
)

pred_distribution = (
    external_results_df["pred_class"]
    .value_counts()
    .reindex(EVAL_CLASSES, fill_value=0)
)

distribution_df = pd.DataFrame({
    "True_Count": true_distribution,
    "Predicted_Count": pred_distribution
})

distribution_df["True_Percent"] = (
    distribution_df["True_Count"]
    / len(external_results_df)
    * 100
).round(2)

distribution_df["Predicted_Percent"] = (
    distribution_df["Predicted_Count"]
    / len(external_results_df)
    * 100
).round(2)

distribution_df

,True_Count,Predicted_Count,True_Percent,Predicted_Percent
Both,41,150,1.99,7.29
Crackle,476,380,23.13,18.46
Normal,1358,825,65.99,40.09
Wheeze,183,703,8.89,34.16


In [43]:
from pathlib import Path

ICBHI_METADATA_DIR = Path(
    r"D:\2026\final_project\data_processed\metadata"
)

for p in sorted(ICBHI_METADATA_DIR.iterdir()):
    print(p.name)

ICBHI_cycles_4k_4sec_metadata.csv
ICBHI_cycles_4k_4sec_metadata_QA.csv
ICBHI_cycles_4k_4sec_QA.csv
ICBHI_patient_level_split_seed777_config.csv
ICBHI_patient_level_split_seed777_v1.csv
ICBHI_patient_level_split_v1.csv


In [44]:
from pathlib import Path
import pandas as pd

ICBHI_METADATA_DIR = Path(
    r"D:\2026\final_project\data_processed\metadata"
)

cycle_meta_path = (
    ICBHI_METADATA_DIR /
    "ICBHI_cycles_4k_4sec_metadata_QA.csv"
)

split_path = (
    ICBHI_METADATA_DIR /
    "ICBHI_patient_level_split_seed777_v1.csv"
)

icbhi_meta = pd.read_csv(cycle_meta_path)
icbhi_split = pd.read_csv(split_path)

print("=== ICBHI cycle metadata ===")
print("shape:", icbhi_meta.shape)
print("columns:")
print(icbhi_meta.columns.tolist())

print("\nFirst 3 rows:")
display(icbhi_meta.head(3))

print("\n=== Patient-level split ===")
print("shape:", icbhi_split.shape)
print("columns:")
print(icbhi_split.columns.tolist())

print("\nFirst 10 rows:")
display(icbhi_split.head(10))

=== ICBHI cycle metadata ===
shape: (6898, 20)
columns:
['patient_id', 'source_wav', 'cycle_id', 'start', 'end', 'original_duration', 'crackle', 'wheeze', 'label', 'original_sr', 'target_sr', 'normalization_method', 'processed_samples', 'processed_duration', 'processed_file', 'use_for_model', 'processed_name', 'near_fullscale_ratio', 'rms', 'quality_flag']

First 3 rows:


,patient_id,source_wav,cycle_id,start,end,original_duration,crackle,wheeze,label,original_sr,target_sr,normalization_method,processed_samples,processed_duration,processed_file,use_for_model,processed_name,near_fullscale_ratio,rms,quality_flag
0,101,101_1b1_Al_sc_Meditron.wav,0,0.036,0.579,0.543,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle00_Normal.wav,0.0,0.029413,OK
1,101,101_1b1_Al_sc_Meditron.wav,1,0.579,2.450,1.871,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle01_Normal.wav,0.0,0.050298,OK
2,101,101_1b1_Al_sc_Meditron.wav,2,2.450,3.893,1.443,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle02_Normal.wav,0.0,0.042131,OK



=== Patient-level split ===
shape: (6896, 21)
columns:
['patient_id', 'source_wav', 'cycle_id', 'start', 'end', 'original_duration', 'crackle', 'wheeze', 'label', 'original_sr', 'target_sr', 'normalization_method', 'processed_samples', 'processed_duration', 'processed_file', 'use_for_model', 'processed_name', 'near_fullscale_ratio', 'rms', 'quality_flag', 'split']

First 10 rows:


,patient_id,source_wav,cycle_id,start,end,original_duration,crackle,wheeze,label,original_sr,...,normalization_method,processed_samples,processed_duration,processed_file,use_for_model,processed_name,near_fullscale_ratio,rms,quality_flag,split
0,101,101_1b1_Al_sc_Meditron.wav,0,0.036,0.579,0.543,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle00_Normal.wav,0.0,0.029413,OK,train
1,101,101_1b1_Al_sc_Meditron.wav,1,0.579,2.450,1.871,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle01_Normal.wav,0.0,0.050298,OK,train
2,101,101_1b1_Al_sc_Meditron.wav,2,2.450,3.893,1.443,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle02_Normal.wav,0.0,0.042131,OK,train
3,101,101_1b1_Al_sc_Meditron.wav,3,3.893,5.793,1.900,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle03_Normal.wav,0.0,0.044338,OK,train
4,101,101_1b1_Al_sc_Meditron.wav,4,5.793,7.521,1.728,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle04_Normal.wav,0.0,0.037884,OK,train
5,101,101_1b1_Al_sc_Meditron.wav,5,7.521,9.279,1.758,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle05_Normal.wav,0.0,0.036922,OK,train
6,101,101_1b1_Al_sc_Meditron.wav,6,9.279,11.150,1.871,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle06_Normal.wav,0.0,0.039681,OK,train
7,101,101_1b1_Al_sc_Meditron.wav,7,11.150,13.036,1.886,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle07_Normal.wav,0.0,0.035533,OK,train
8,101,101_1b1_Al_sc_Meditron.wav,8,13.036,14.721,1.685,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle08_Normal.wav,0.0,0.031531,OK,train
9,101,101_1b1_Al_sc_Meditron.wav,9,14.721,16.707,1.986,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle09_Normal.wav,0.0,0.041491,OK,train


In [45]:
# metadata의 processed_file 전체 경로에서 파일명만 추출
icbhi_meta["processed_name"] = (
    icbhi_meta["processed_file"]
    .apply(lambda x: Path(x).name)
)

# split 정보와 결합
icbhi_merged = icbhi_meta.merge(
    icbhi_split[
        ["processed_name", "quality_flag", "split"]
    ],
    on="processed_name",
    how="left"
)

print("Merged rows:", len(icbhi_merged))
print("\nSplit counts:")
print(icbhi_merged["split"].value_counts(dropna=False))

print("\nMissing split:")
print(icbhi_merged["split"].isna().sum())

Merged rows: 6898

Split counts:
split
train         4737
validation    1141
test          1018
NaN              2
Name: count, dtype: int64

Missing split:
2


In [46]:
icbhi_test = icbhi_merged[
    (icbhi_merged["split"] == "test") &
    (icbhi_merged["use_for_model"] == True)
].copy()

print("ICBHI test cycles:", len(icbhi_test))

print("\nNormalization method:")
print(
    icbhi_test["normalization_method"]
    .value_counts()
)

print("\nOriginal duration:")
display(
    icbhi_test["original_duration"]
    .describe()
)

ICBHI test cycles: 1018

Normalization method:
normalization_method
padding        879
center_crop    137
none             2
Name: count, dtype: int64

Original duration:


count    1018.000000
mean        2.550434
std         1.165445
min         0.315000
25%         1.634925
50%         2.452000
75%         3.209675
max         6.328000
Name: original_duration, dtype: float64

In [47]:
icbhi_padding_count = (
    icbhi_test["normalization_method"] == "padding"
).sum()

icbhi_crop_count = (
    icbhi_test["normalization_method"] == "crop"
).sum()

icbhi_none_count = (
    icbhi_test["normalization_method"] == "none"
).sum()

hf_padding_count = (
    hf_test_metadata["normalization_method"]
    == "symmetric_zero_padding"
).sum()

input_comparison_df = pd.DataFrame({
    "Item": [
        "Number of cycles",
        "Mean original duration (sec)",
        "Median original duration (sec)",
        "Minimum duration (sec)",
        "Maximum duration (sec)",
        "Padding count",
        "Padding percent",
        "Crop count",
        "Crop percent"
    ],
    "ICBHI_Internal_Test": [
        len(icbhi_test),
        icbhi_test["original_duration"].mean(),
        icbhi_test["original_duration"].median(),
        icbhi_test["original_duration"].min(),
        icbhi_test["original_duration"].max(),
        icbhi_padding_count,
        icbhi_padding_count / len(icbhi_test) * 100,
        icbhi_crop_count,
        icbhi_crop_count / len(icbhi_test) * 100
    ],
    "HF_Lung_External_Test": [
        len(hf_test_metadata),
        hf_test_metadata["original_duration_sec"].mean(),
        hf_test_metadata["original_duration_sec"].median(),
        hf_test_metadata["original_duration_sec"].min(),
        hf_test_metadata["original_duration_sec"].max(),
        hf_padding_count,
        hf_padding_count / len(hf_test_metadata) * 100,
        0,
        0.0
    ]
})

input_comparison_df

,Item,ICBHI_Internal_Test,HF_Lung_External_Test
0,Number of cycles,1018.000000,2058.00000
1,Mean original duration (sec),2.550434,2.06181
2,Median original duration (sec),2.452000,1.95500
3,Minimum duration (sec),0.315000,0.86500
4,Maximum duration (sec),6.328000,3.95400
5,Padding count,879.000000,2058.00000
6,Padding percent,86.345776,100.00000
7,Crop count,0.000000,0.00000
8,Crop percent,0.000000,0.00000


In [48]:
icbhi_padding_count = (
    icbhi_test["normalization_method"] == "padding"
).sum()

icbhi_crop_count = (
    icbhi_test["normalization_method"] == "center_crop"
).sum()

icbhi_none_count = (
    icbhi_test["normalization_method"] == "none"
).sum()

hf_padding_count = (
    hf_test_metadata["normalization_method"]
    == "symmetric_zero_padding"
).sum()


input_comparison_df = pd.DataFrame({
    "Item": [
        "Number of cycles",
        "Mean original duration (sec)",
        "Median original duration (sec)",
        "Minimum duration (sec)",
        "Maximum duration (sec)",
        "Padding count",
        "Padding percent",
        "Center crop count",
        "Center crop percent",
        "No normalization count"
    ],
    "ICBHI_Internal_Test": [
        len(icbhi_test),
        icbhi_test["original_duration"].mean(),
        icbhi_test["original_duration"].median(),
        icbhi_test["original_duration"].min(),
        icbhi_test["original_duration"].max(),
        icbhi_padding_count,
        icbhi_padding_count / len(icbhi_test) * 100,
        icbhi_crop_count,
        icbhi_crop_count / len(icbhi_test) * 100,
        icbhi_none_count
    ],
    "HF_Lung_External_Test": [
        len(hf_test_metadata),
        hf_test_metadata["original_duration_sec"].mean(),
        hf_test_metadata["original_duration_sec"].median(),
        hf_test_metadata["original_duration_sec"].min(),
        hf_test_metadata["original_duration_sec"].max(),
        hf_padding_count,
        hf_padding_count / len(hf_test_metadata) * 100,
        0,
        0.0,
        0
    ]
})

input_comparison_df

,Item,ICBHI_Internal_Test,HF_Lung_External_Test
0,Number of cycles,1018.000000,2058.00000
1,Mean original duration (sec),2.550434,2.06181
2,Median original duration (sec),2.452000,1.95500
3,Minimum duration (sec),0.315000,0.86500
4,Maximum duration (sec),6.328000,3.95400
5,Padding count,879.000000,2058.00000
6,Padding percent,86.345776,100.00000
7,Center crop count,137.000000,0.00000
8,Center crop percent,13.457760,0.00000
9,No normalization count,2.000000,0.00000


In [49]:
icbhi_test["effective_signal_sec"] = (
    icbhi_test["original_duration"]
    .clip(upper=4.0)
)

icbhi_test["signal_ratio"] = (
    icbhi_test["effective_signal_sec"] / 4.0
)


# HF_Lung은 모두 4초 이하
hf_signal_analysis = hf_test_metadata.copy()

hf_signal_analysis["signal_ratio"] = (
    hf_signal_analysis["original_duration_sec"] / 4.0
)


signal_ratio_df = pd.DataFrame({
    "Dataset": [
        "ICBHI Internal Test",
        "HF_Lung External Test"
    ],
    "Mean_Signal_Ratio": [
        icbhi_test["signal_ratio"].mean(),
        hf_signal_analysis["signal_ratio"].mean()
    ],
    "Median_Signal_Ratio": [
        icbhi_test["signal_ratio"].median(),
        hf_signal_analysis["signal_ratio"].median()
    ],
    "Mean_Padding_Ratio": [
        1 - icbhi_test["signal_ratio"].mean(),
        1 - hf_signal_analysis["signal_ratio"].mean()
    ]
})

signal_ratio_df

,Dataset,Mean_Signal_Ratio,Median_Signal_Ratio,Mean_Padding_Ratio
0,ICBHI Internal Test,0.615132,0.61300,0.384868
1,HF_Lung External Test,0.515453,0.48875,0.484547


In [50]:
print("ICBHI signal ratio")
display(icbhi_test["signal_ratio"].describe())

print("\nHF_Lung signal ratio")
display(hf_signal_analysis["signal_ratio"].describe())

ICBHI signal ratio


count    1018.000000
mean        0.615132
std         0.248122
min         0.078750
25%         0.408731
50%         0.613000
75%         0.802419
max         1.000000
Name: signal_ratio, dtype: float64


HF_Lung signal ratio


count    2058.000000
mean        0.515453
std         0.125823
min         0.216250
25%         0.432000
50%         0.488750
75%         0.566438
max         0.988500
Name: signal_ratio, dtype: float64

In [51]:
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from tqdm.auto import tqdm
from pathlib import Path

In [52]:
# 기존 Mel 설정 재사용
SR = 4000
N_FFT = 512
HOP_LENGTH = 128
N_MELS = 128
F_MIN = 0
F_MAX = 2000

mel_transform_compare = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=F_MIN,
    f_max=F_MAX,
    power=2.0
)

In [53]:
def acoustic_features(wav_path):
    audio, sr = sf.read(wav_path, dtype="float32")

    if sr != SR:
        raise ValueError(f"Unexpected SR: {wav_path} / {sr}")

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # RMS
    rms = float(np.sqrt(np.mean(audio ** 2)))

    # Peak amplitude
    peak = float(np.max(np.abs(audio)))

    # Mel
    waveform = torch.tensor(audio).float().unsqueeze(0)

    mel = mel_transform_compare(waveform)

    mel_db = 10.0 * torch.log10(
        torch.clamp(mel, min=1e-10)
    )

    mel_db = mel_db - mel_db.max()
    mel_db = torch.clamp(
        mel_db,
        min=-80.0,
        max=0.0
    )

    mean_mel_db = float(mel_db.mean().item())

    return {
        "rms": rms,
        "peak": peak,
        "mean_mel_db": mean_mel_db
    }

In [54]:
icbhi_acoustic_rows = []

for _, row in tqdm(
    icbhi_test.iterrows(),
    total=len(icbhi_test),
    desc="ICBHI acoustic analysis"
):

    wav_path = Path(row["processed_file"])

    feat = acoustic_features(wav_path)

    icbhi_acoustic_rows.append({
        "dataset": "ICBHI",
        "class": row["label"],
        "file": wav_path.name,
        **feat
    })

icbhi_acoustic_df = pd.DataFrame(
    icbhi_acoustic_rows
)

icbhi_acoustic_df.head()

ICBHI acoustic analysis: 100%|██████████| 1018/1018 [00:06<00:00, 146.96it/s]


,dataset,class,file,rms,peak,mean_mel_db
0,ICBHI,Crackle,107_2b3_Al_mc_AKGC417L_cycle00_Crackle.wav,0.277262,1.001838,-70.999443
1,ICBHI,Crackle,107_2b3_Al_mc_AKGC417L_cycle01_Crackle.wav,0.295840,1.003360,-64.401840
2,ICBHI,Crackle,107_2b3_Al_mc_AKGC417L_cycle02_Crackle.wav,0.311530,1.003860,-65.323883
3,ICBHI,Crackle,107_2b3_Al_mc_AKGC417L_cycle03_Crackle.wav,0.319331,1.087815,-64.501068
4,ICBHI,Crackle,107_2b3_Al_mc_AKGC417L_cycle04_Crackle.wav,0.330064,1.086960,-65.011536


In [55]:
hf_acoustic_rows = []

for _, row in tqdm(
    hf_test_metadata.iterrows(),
    total=len(hf_test_metadata),
    desc="HF_Lung acoustic analysis"
):

    wav_path = (
        TEST_OUTPUT /
        row["class"] /
        row["output_file"]
    )

    feat = acoustic_features(wav_path)

    hf_acoustic_rows.append({
        "dataset": "HF_Lung",
        "class": row["class"],
        "file": wav_path.name,
        **feat
    })

hf_acoustic_df = pd.DataFrame(
    hf_acoustic_rows
)

hf_acoustic_df.head()

HF_Lung acoustic analysis: 100%|██████████| 2058/2058 [00:02<00:00, 697.15it/s]


,dataset,class,file,rms,peak,mean_mel_db
0,HF_Lung,Normal,steth_20190801_09_51_07_cycle001_Normal.wav,0.021275,0.220551,-59.520744
1,HF_Lung,Normal,steth_20190801_09_51_07_cycle002_Normal.wav,0.025825,0.355072,-60.382946
2,HF_Lung,Normal,steth_20190801_09_51_07_cycle003_Normal.wav,0.024579,0.263336,-62.219593
3,HF_Lung,Normal,steth_20190801_09_51_07_cycle004_Normal.wav,0.021954,0.250275,-62.111595
4,HF_Lung,Normal,steth_20190801_09_51_07_cycle005_Normal.wav,0.023604,0.290070,-58.608822


In [56]:
acoustic_summary_df = pd.DataFrame({
    "Metric": [
        "RMS mean",
        "RMS median",
        "Peak mean",
        "Peak median",
        "Mean Mel dB mean",
        "Mean Mel dB median"
    ],
    "ICBHI_Internal_Test": [
        icbhi_acoustic_df["rms"].mean(),
        icbhi_acoustic_df["rms"].median(),
        icbhi_acoustic_df["peak"].mean(),
        icbhi_acoustic_df["peak"].median(),
        icbhi_acoustic_df["mean_mel_db"].mean(),
        icbhi_acoustic_df["mean_mel_db"].median()
    ],
    "HF_Lung_External_Test": [
        hf_acoustic_df["rms"].mean(),
        hf_acoustic_df["rms"].median(),
        hf_acoustic_df["peak"].mean(),
        hf_acoustic_df["peak"].median(),
        hf_acoustic_df["mean_mel_db"].mean(),
        hf_acoustic_df["mean_mel_db"].median()
    ]
})

acoustic_summary_df

,Metric,ICBHI_Internal_Test,HF_Lung_External_Test
0,RMS mean,0.147929,0.018519
1,RMS median,0.130929,0.011973
2,Peak mean,0.659983,0.249773
3,Peak median,0.726245,0.167526
4,Mean Mel dB mean,-64.699518,-57.142186
5,Mean Mel dB median,-65.410645,-58.392286


In [57]:
icbhi_class_acoustic = (
    icbhi_acoustic_df
    .groupby("class")[["rms", "peak", "mean_mel_db"]]
    .mean()
    .round(6)
)

hf_class_acoustic = (
    hf_acoustic_df
    .groupby("class")[["rms", "peak", "mean_mel_db"]]
    .mean()
    .round(6)
)

print("ICBHI")
display(icbhi_class_acoustic)

print("HF_Lung")
display(hf_class_acoustic)

ICBHI


,rms,peak,mean_mel_db
class,,,
Both,0.211960,0.809802,-64.985350
Crackle,0.203188,0.793478,-60.477834
Normal,0.115388,0.580901,-66.615007
Wheeze,0.128422,0.616728,-65.523281


HF_Lung


,rms,peak,mean_mel_db
class,,,
Both,0.027358,0.395766,-61.124776
Crackle,0.024496,0.332264,-58.219104
Normal,0.015637,0.215060,-56.622292
Wheeze,0.022376,0.260092,-57.306758


In [58]:
def nonzero_rms(audio):
    audio = np.asarray(audio)

    # zero-padding 부분 제외
    nonzero = audio[np.abs(audio) > 1e-8]

    if len(nonzero) == 0:
        return np.nan

    return float(
        np.sqrt(np.mean(nonzero ** 2))
    )


def nonzero_peak(audio):
    audio = np.asarray(audio)

    nonzero = audio[np.abs(audio) > 1e-8]

    if len(nonzero) == 0:
        return np.nan

    return float(
        np.max(np.abs(nonzero))
    )

In [59]:
icbhi_nonzero_rows = []

for _, row in tqdm(
    icbhi_test.iterrows(),
    total=len(icbhi_test),
    desc="ICBHI nonzero analysis"
):

    audio, sr = sf.read(
        row["processed_file"],
        dtype="float32"
    )

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    icbhi_nonzero_rows.append({
        "class": row["label"],
        "rms_nonzero": nonzero_rms(audio),
        "peak_nonzero": nonzero_peak(audio)
    })


icbhi_nonzero_df = pd.DataFrame(
    icbhi_nonzero_rows
)

ICBHI nonzero analysis: 100%|██████████| 1018/1018 [00:00<00:00, 2143.15it/s]


In [60]:
hf_nonzero_rows = []

for _, row in tqdm(
    hf_test_metadata.iterrows(),
    total=len(hf_test_metadata),
    desc="HF_Lung nonzero analysis"
):

    wav_path = (
        TEST_OUTPUT /
        row["class"] /
        row["output_file"]
    )

    audio, sr = sf.read(
        wav_path,
        dtype="float32"
    )

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    hf_nonzero_rows.append({
        "class": row["class"],
        "rms_nonzero": nonzero_rms(audio),
        "peak_nonzero": nonzero_peak(audio)
    })


hf_nonzero_df = pd.DataFrame(
    hf_nonzero_rows
)

HF_Lung nonzero analysis: 100%|██████████| 2058/2058 [00:00<00:00, 2091.46it/s]


In [61]:
nonzero_summary_df = pd.DataFrame({
    "Metric": [
        "Nonzero RMS mean",
        "Nonzero RMS median",
        "Nonzero Peak mean",
        "Nonzero Peak median"
    ],

    "ICBHI_Internal_Test": [
        icbhi_nonzero_df["rms_nonzero"].mean(),
        icbhi_nonzero_df["rms_nonzero"].median(),
        icbhi_nonzero_df["peak_nonzero"].mean(),
        icbhi_nonzero_df["peak_nonzero"].median()
    ],

    "HF_Lung_External_Test": [
        hf_nonzero_df["rms_nonzero"].mean(),
        hf_nonzero_df["rms_nonzero"].median(),
        hf_nonzero_df["peak_nonzero"].mean(),
        hf_nonzero_df["peak_nonzero"].median()
    ]
})

nonzero_summary_df

,Metric,ICBHI_Internal_Test,HF_Lung_External_Test
0,Nonzero RMS mean,0.195480,0.026254
1,Nonzero RMS median,0.170455,0.016972
2,Nonzero Peak mean,0.659983,0.249773
3,Nonzero Peak median,0.726245,0.167526


In [3]:
from pathlib import Path
from collections import Counter
import json

# =========================================================
# SPRSound BioCAS2022 경로
# 본인의 실제 경로로 수정하세요.
# =========================================================

ROOT = Path(r"D:\2026\final_project\external_data\SPRSound_BioCAS2022_raw\BioCAS2022")

print("=" * 70)
print("SPRSound BioCAS2022 구조 확인")
print("=" * 70)

folders = [
    "train2022_wav",
    "train2022_json",
    "test2022_wav",
    "test2022_json",
]

for folder in folders:
    path = ROOT / folder

    print(f"\n[{folder}]")

    if not path.exists():
        print("  → 폴더 없음")
        continue

    wav_count = len(list(path.rglob("*.wav")))
    json_count = len(list(path.rglob("*.json")))

    print(f"  경로      : {path}")
    print(f"  WAV 수    : {wav_count}")
    print(f"  JSON 수   : {json_count}")


# =========================================================
# JSON annotation label 확인
# =========================================================

print("\n" + "=" * 70)
print("Event annotation label 확인")
print("=" * 70)

event_counter = Counter()
record_counter = Counter()

json_files = list((ROOT / "test2022_json").rglob("*.json"))

for json_file in json_files:

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    if "record_annotation" in data:
        record_counter[data["record_annotation"]] += 1

    for event in data.get("event_annotation", []):
        event_counter[event.get("type")] += 1


print("\n[Record-level labels]")
for label, n in record_counter.most_common():
    print(f"{label:20s}: {n}")

print("\n[Event-level labels]")
for label, n in event_counter.most_common():
    print(f"{label:20s}: {n}")


print("\n" + "=" * 70)
print("확인 완료")
print("=" * 70)

SPRSound BioCAS2022 구조 확인

[train2022_wav]
  경로      : D:\2026\final_project\external_data\SPRSound_BioCAS2022_raw\BioCAS2022\train2022_wav
  WAV 수    : 1949
  JSON 수   : 0

[train2022_json]
  경로      : D:\2026\final_project\external_data\SPRSound_BioCAS2022_raw\BioCAS2022\train2022_json
  WAV 수    : 0
  JSON 수   : 1949

[test2022_wav]
  경로      : D:\2026\final_project\external_data\SPRSound_BioCAS2022_raw\BioCAS2022\test2022_wav
  WAV 수    : 734
  JSON 수   : 0

[test2022_json]
  경로      : D:\2026\final_project\external_data\SPRSound_BioCAS2022_raw\BioCAS2022\test2022_json
  WAV 수    : 0
  JSON 수   : 734

Event annotation label 확인

[Record-level labels]
Normal              : 482
CAS                 : 107
DAS                 : 99
CAS & DAS           : 36
Poor Quality        : 10

[Event-level labels]
Normal              : 1728
Wheeze              : 413
Fine Crackle        : 255
Coarse Crackle      : 17
Rhonchi             : 14
Wheeze+Crackle      : 4
Stridor             : 2

확인 완료


In [4]:
from pathlib import Path
from collections import Counter
import json

# =========================================================
# 1. test JSON 파일 확인
# =========================================================

json_files = sorted((ROOT / "test2022_json").rglob("*.json"))

print("=" * 70)
print("SPRSound BioCAS2022 JSON 구조 확인")
print("=" * 70)

print(f"\nTest JSON 파일 수: {len(json_files)}")

# 첫 번째 JSON 확인
sample_file = json_files[0]

print(f"\n[샘플 파일]")
print(sample_file.name)

with open(sample_file, "r", encoding="utf-8") as f:
    sample = json.load(f)

print("\n[최상위 key]")
print(sample.keys())

print("\n[첫 번째 JSON 전체 구조]")
print(json.dumps(sample, indent=2, ensure_ascii=False))


# =========================================================
# 2. 전체 label 확인
# =========================================================

record_labels = Counter()
event_labels = Counter()

for jf in json_files:

    with open(jf, "r", encoding="utf-8") as f:
        data = json.load(f)

    # record-level
    if "record_annotation" in data:
        record_labels[str(data["record_annotation"])] += 1

    # event-level
    events = data.get("event_annotation", [])

    if isinstance(events, list):
        for event in events:

            if isinstance(event, dict):

                # 실제 label field 후보 확인
                label = (
                    event.get("type")
                    or event.get("label")
                    or event.get("annotation")
                    or event.get("event")
                )

                if label is not None:
                    event_labels[str(label)] += 1


print("\n" + "=" * 70)
print("Record-level label")
print("=" * 70)

if record_labels:
    for label, n in record_labels.most_common():
        print(f"{label:25s} : {n}")
else:
    print("record_annotation 없음")


print("\n" + "=" * 70)
print("Event-level label")
print("=" * 70)

if event_labels:
    for label, n in event_labels.most_common():
        print(f"{label:25s} : {n}")
else:
    print("현재 코드에서 event label을 찾지 못했습니다.")


print("\n" + "=" * 70)
print("JSON 구조 확인 완료")
print("=" * 70)

SPRSound BioCAS2022 JSON 구조 확인

Test JSON 파일 수: 734

[샘플 파일]
40512331_8.1_1_p1_3544.json

[최상위 key]
dict_keys(['record_annotation', 'event_annotation'])

[첫 번째 JSON 전체 구조]
{
  "record_annotation": "Poor Quality",
  "event_annotation": []
}

Record-level label
Normal                    : 482
CAS                       : 107
DAS                       : 99
CAS & DAS                 : 36
Poor Quality              : 10

Event-level label
Normal                    : 1728
Wheeze                    : 413
Fine Crackle              : 255
Coarse Crackle            : 17
Rhonchi                   : 14
Wheeze+Crackle            : 4
Stridor                   : 2

JSON 구조 확인 완료


In [5]:
from collections import Counter
import json

event_labels = Counter()
record_labels = Counter()

first_event_example = None
first_event_file = None

json_files = sorted((ROOT / "test2022_json").rglob("*.json"))

for jf in json_files:

    with open(jf, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Record-level
    record_label = data.get("record_annotation")
    if record_label is not None:
        record_labels[str(record_label)] += 1

    # Event-level
    events = data.get("event_annotation", [])

    for event in events:
        if isinstance(event, dict):

            if first_event_example is None:
                first_event_example = event
                first_event_file = jf.name

            # 어떤 key가 실제 label인지 모두 확인
            label = (
                event.get("type")
                or event.get("label")
                or event.get("annotation")
                or event.get("event")
            )

            if label is not None:
                event_labels[str(label)] += 1


print("=" * 70)
print("RECORD-LEVEL")
print("=" * 70)

for k, v in record_labels.most_common():
    print(f"{k:25s}: {v}")

print(f"\n총 record 수: {sum(record_labels.values())}")


print("\n" + "=" * 70)
print("첫 번째 EVENT 구조")
print("=" * 70)

print("파일:", first_event_file)
print(first_event_example)


print("\n" + "=" * 70)
print("EVENT-LEVEL LABEL")
print("=" * 70)

for k, v in event_labels.most_common():
    print(f"{k:25s}: {v}")

print(f"\n총 event 수: {sum(event_labels.values())}")

RECORD-LEVEL
Normal                   : 482
CAS                      : 107
DAS                      : 99
CAS & DAS                : 36
Poor Quality             : 10

총 record 수: 734

첫 번째 EVENT 구조
파일: 40512331_8.1_1_p1_3548.json
{'start': '17', 'end': '1623', 'type': 'Normal'}

EVENT-LEVEL LABEL
Normal                   : 1728
Wheeze                   : 413
Fine Crackle             : 255
Coarse Crackle           : 17
Rhonchi                  : 14
Wheeze+Crackle           : 4
Stridor                  : 2

총 event 수: 2433


In [6]:
from pathlib import Path
import json
import pandas as pd

# =========================================================
# 경로
# =========================================================

TEST_WAV_DIR = ROOT / "test2022_wav"
TEST_JSON_DIR = ROOT / "test2022_json"

PROJECT_ROOT = Path(r"D:\2026\final_project")
MANIFEST_DIR = PROJECT_ROOT / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = MANIFEST_DIR / "sprsound_test_manifest_v1.csv"


# =========================================================
# ICBHI-compatible label mapping
# =========================================================

LABEL_MAP = {
    "Normal": "Normal",
    "Fine Crackle": "Crackle",
    "Coarse Crackle": "Crackle",
    "Wheeze": "Wheeze",
    "Wheeze+Crackle": "Both",
}

EXCLUDED_EVENT_LABELS = {
    "Rhonchi",
    "Stridor",
}


# =========================================================
# Manifest 생성
# =========================================================

rows = []

json_files = sorted(TEST_JSON_DIR.rglob("*.json"))

for json_file in json_files:

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    record_label = data.get("record_annotation")

    # JSON과 동일 stem의 WAV 찾기
    wav_file = TEST_WAV_DIR / f"{json_file.stem}.wav"

    # -----------------------------------------------------
    # Poor Quality는 recording-level exclusion
    # -----------------------------------------------------
    if record_label == "Poor Quality":
        continue

    events = data.get("event_annotation", [])

    for event_idx, event in enumerate(events):

        original_label = event.get("type")
        start_raw = event.get("start")
        end_raw = event.get("end")

        if original_label in LABEL_MAP:
            mapped_label = LABEL_MAP[original_label]
            include_primary = True

        elif original_label in EXCLUDED_EVENT_LABELS:
            mapped_label = None
            include_primary = False

        else:
            mapped_label = None
            include_primary = False

        rows.append({
            "record_id": json_file.stem,
            "event_id": event_idx,
            "wav_path": str(wav_file),
            "json_path": str(json_file),
            "record_annotation": record_label,
            "event_label_original": original_label,
            "event_label_icbhi": mapped_label,
            "start_raw": start_raw,
            "end_raw": end_raw,
            "include_primary": include_primary,
        })


df = pd.DataFrame(rows)

df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)


# =========================================================
# 검증
# =========================================================

print("=" * 70)
print("SPRSound Test Manifest 생성 완료")
print("=" * 70)

print("\n저장 위치:")
print(OUTPUT_CSV)

print("\n전체 event:")
print(len(df))

print("\nOriginal label:")
print(df["event_label_original"].value_counts())

print("\nICBHI-compatible primary events:")
primary_df = df[df["include_primary"] == True]

print(primary_df["event_label_icbhi"].value_counts())

print("\nPrimary event 수:")
print(len(primary_df))

print("\nExcluded event:")
print(df[df["include_primary"] == False]["event_label_original"].value_counts())

print("\nWAV 파일 존재 여부:")
print(df["wav_path"].apply(lambda x: Path(x).exists()).value_counts())

print("\n앞 10개:")
display(df.head(10))

SPRSound Test Manifest 생성 완료

저장 위치:
D:\2026\final_project\manifests\sprsound_test_manifest_v1.csv

전체 event:
2433

Original label:
event_label_original
Normal            1728
Wheeze             413
Fine Crackle       255
Coarse Crackle      17
Rhonchi             14
Wheeze+Crackle       4
Stridor              2
Name: count, dtype: int64

ICBHI-compatible primary events:
event_label_icbhi
Normal     1728
Wheeze      413
Crackle     272
Both          4
Name: count, dtype: int64

Primary event 수:
2417

Excluded event:
event_label_original
Rhonchi    14
Stridor     2
Name: count, dtype: int64

WAV 파일 존재 여부:
wav_path
True    2433
Name: count, dtype: int64

앞 10개:


,record_id,event_id,wav_path,json_path,record_annotation,event_label_original,event_label_icbhi,start_raw,end_raw,include_primary
0,40512331_8.1_1_p1_3548,0,D:\2026\final_project\external_data\SPRSound_B...,D:\2026\final_project\external_data\SPRSound_B...,Normal,Normal,Normal,17,1623,True
1,40512331_8.1_1_p1_3548,1,D:\2026\final_project\external_data\SPRSound_B...,D:\2026\final_project\external_data\SPRSound_B...,Normal,Normal,Normal,1623,2878,True
2,40512331_8.1_1_p1_3552,0,D:\2026\final_project\external_data\SPRSound_B...,D:\2026\final_project\external_data\SPRSound_B...,Normal,Normal,Normal,4878,6847,True
3,40512331_8.1_1_p1_3552,1,D:\2026\final_project\external_data\SPRSound_B...,D:\2026\final_project\external_data\SPRSound_B...,Normal,Normal,Normal,8007,9194,True
4,40512331_8.1_1_p1_3552,2,D:\2026\final_project\external_data\SPRSound_B...,D:\2026\final_project\external_data\SPRSound_B...,Normal,Normal,Normal,6847,8007,True
5,40512331_8.1_1_p2_3545,0,D:\2026\final_project\external_data\SPRSound_B...,D:\2026\final_project\external_data\SPRSound_B...,Normal,Normal,Normal,6510,8143,True
6,40512331_8.1_1_p2_3545,1,D:\2026\final_project\external_data\SPRSound_B...,D:\2026\final_project\external_data\SPRSound_B...,Normal,Normal,Normal,8143,9206,True
7,40512331_8.1_1_p4_3543,0,D:\2026\final_project\external_data\SPRSound_B...,D:\2026\final_project\external_data\SPRSound_B...,Normal,Normal,Normal,4283,6047,True
8,40512331_8.1_1_p4_3547,0,D:\2026\final_project\external_data\SPRSound_B...,D:\2026\final_project\external_data\SPRSound_B...,Normal,Normal,Normal,245,1841,True
9,40512331_8.1_1_p4_3551,0,D:\2026\final_project\external_data\SPRSound_B...,D:\2026\final_project\external_data\SPRSound_B...,Normal,Normal,Normal,8365,9171,True


In [7]:
from pathlib import Path
import soundfile as sf
import pandas as pd

print("=" * 80)
print("SPRSound annotation 시간 단위 검증")
print("=" * 80)

# 서로 다른 recording 10개 선택
sample_records = (
    df[["record_id", "wav_path"]]
    .drop_duplicates()
    .head(10)
)

results = []

for _, row in sample_records.iterrows():

    record_id = row["record_id"]
    wav_path = Path(row["wav_path"])

    # WAV 정보
    info = sf.info(str(wav_path))

    sr = info.samplerate
    frames = info.frames
    duration_sec = frames / sr

    # 해당 recording의 모든 annotation
    sub = df[df["record_id"] == record_id]

    max_end_raw = pd.to_numeric(
        sub["end_raw"],
        errors="coerce"
    ).max()

    results.append({
        "record_id": record_id,
        "sample_rate": sr,
        "frames": frames,
        "wav_duration_sec": round(duration_sec, 3),
        "max_end_raw": max_end_raw,
        "max_end_div_1000": round(max_end_raw / 1000, 3),
        "difference_sec": round(
            duration_sec - (max_end_raw / 1000),
            3
        ),
    })


check_df = pd.DataFrame(results)

display(check_df)

print("\n" + "=" * 80)

print("Sampling rate 분포:")
print(check_df["sample_rate"].value_counts())

print("\nWAV duration 범위:")
print(
    check_df["wav_duration_sec"].min(),
    "~",
    check_df["wav_duration_sec"].max(),
    "sec"
)

print("\nmax(end_raw / 1000) 범위:")
print(
    check_df["max_end_div_1000"].min(),
    "~",
    check_df["max_end_div_1000"].max(),
    "sec"
)

SPRSound annotation 시간 단위 검증


,record_id,sample_rate,frames,wav_duration_sec,max_end_raw,max_end_div_1000,difference_sec
0,40512331_8.1_1_p1_3548,8000,73728,9.216,2878,2.878,6.338
1,40512331_8.1_1_p1_3552,8000,73728,9.216,9194,9.194,0.022
2,40512331_8.1_1_p2_3545,8000,73728,9.216,9206,9.206,0.010
3,40512331_8.1_1_p4_3543,8000,73728,9.216,6047,6.047,3.169
4,40512331_8.1_1_p4_3547,8000,73728,9.216,1841,1.841,7.375
5,40512331_8.1_1_p4_3551,8000,73728,9.216,9171,9.171,0.045
6,40888395_3.4_0_p1_1146,8000,73728,9.216,8803,8.803,0.413
7,40888395_3.4_0_p3_1136,8000,73728,9.216,7085,7.085,2.131
8,40888395_3.4_0_p4_1165,8000,73728,9.216,8005,8.005,1.211
9,40890405_3.3_0_p1_3652,8000,122880,15.360,15245,15.245,0.115



Sampling rate 분포:
sample_rate
8000    10
Name: count, dtype: int64

WAV duration 범위:
9.216 ~ 15.36 sec

max(end_raw / 1000) 범위:
1.841 ~ 15.245 sec


In [8]:
from pathlib import Path
import pandas as pd
import soundfile as sf

print("=" * 80)
print("SPRSound 전체 Event Timing QC")
print("=" * 80)

qc_rows = []

# recording마다 WAV metadata를 한 번만 읽기
wav_info_cache = {}

for idx, row in df.iterrows():

    wav_path = Path(row["wav_path"])
    record_id = row["record_id"]

    if record_id not in wav_info_cache:

        info = sf.info(str(wav_path))

        wav_info_cache[record_id] = {
            "sample_rate": info.samplerate,
            "frames": info.frames,
            "duration_sec": info.frames / info.samplerate,
        }

    info = wav_info_cache[record_id]

    start_ms = float(row["start_raw"])
    end_ms = float(row["end_raw"])

    start_sec = start_ms / 1000.0
    end_sec = end_ms / 1000.0

    event_duration_sec = end_sec - start_sec

    qc_rows.append({
        "record_id": record_id,
        "event_id": row["event_id"],
        "label": row["event_label_original"],

        "sample_rate": info["sample_rate"],
        "wav_duration_sec": info["duration_sec"],

        "start_ms": start_ms,
        "end_ms": end_ms,

        "start_sec": start_sec,
        "end_sec": end_sec,
        "event_duration_sec": event_duration_sec,

        "start_negative": start_sec < 0,
        "end_before_start": end_sec <= start_sec,

        # 1 sample 정도의 수치 오차 허용
        "end_over_wav": end_sec > (
            info["duration_sec"] + 1 / info["sample_rate"]
        )
    })


qc_df = pd.DataFrame(qc_rows)


print("\n전체 event 수:")
print(len(qc_df))


print("\nSampling rate:")
print(qc_df["sample_rate"].value_counts())


print("\nTiming 오류:")
print("start < 0       :", qc_df["start_negative"].sum())
print("end <= start    :", qc_df["end_before_start"].sum())
print("end > WAV length:", qc_df["end_over_wav"].sum())


print("\nEvent duration 통계 (sec):")
print(
    qc_df["event_duration_sec"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
    .round(3)
)


print("\n가장 짧은 event 10개:")
display(
    qc_df
    .sort_values("event_duration_sec")
    .head(10)
)


print("\n가장 긴 event 10개:")
display(
    qc_df
    .sort_values("event_duration_sec", ascending=False)
    .head(10)
)

SPRSound 전체 Event Timing QC

전체 event 수:
2433

Sampling rate:
sample_rate
8000    2433
Name: count, dtype: int64

Timing 오류:
start < 0       : 0
end <= start    : 0
end > WAV length: 0

Event duration 통계 (sec):
count    2433.000
mean        1.369
std         0.732
min         0.232
1%          0.315
5%          0.418
25%         0.755
50%         1.287
75%         1.845
95%         2.662
99%         3.407
max         5.665
Name: event_duration_sec, dtype: float64

가장 짧은 event 10개:


,record_id,event_id,label,sample_rate,wav_duration_sec,start_ms,end_ms,start_sec,end_sec,event_duration_sec,start_negative,end_before_start,end_over_wav
2077,64743918_7.0_0_p4_2775,3,Wheeze,8000,9.216,8161.0,8393.0,8.161,8.393,0.232,False,False,False
1235,65114720_0.9_0_p3_3736,7,Wheeze,8000,9.216,4290.0,4532.0,4.290,4.532,0.242,False,False,False
2109,64783073_1.3_0_p2_3277,1,Wheeze,8000,9.216,1997.0,2240.0,1.997,2.240,0.243,False,False,False
912,65049153_0.8_0_p3_3774,4,Normal,8000,15.360,1793.0,2037.0,1.793,2.037,0.244,False,False,False
911,65049153_0.8_0_p3_3774,3,Normal,8000,15.360,2577.0,2828.0,2.577,2.828,0.251,False,False,False
2057,64743918_7.0_0_p3_2614,1,Wheeze,8000,9.216,5559.0,5816.0,5.559,5.816,0.257,False,False,False
1699,41134874_13.2_1_p4_356,0,Wheeze,8000,9.216,6712.0,6971.0,6.712,6.971,0.259,False,False,False
2075,64743918_7.0_0_p4_2775,1,Wheeze,8000,9.216,4624.0,4887.0,4.624,4.887,0.263,False,False,False
1658,41104324_3.3_1_p1_3190,2,Wheeze,8000,9.216,5615.0,5890.0,5.615,5.890,0.275,False,False,False
905,65049153_0.8_0_p3_3761,5,Fine Crackle,8000,15.360,300.0,577.0,0.300,0.577,0.277,False,False,False



가장 긴 event 10개:


,record_id,event_id,label,sample_rate,wav_duration_sec,start_ms,end_ms,start_sec,end_sec,event_duration_sec,start_negative,end_before_start,end_over_wav
1767,41227367_6.7_1_p1_625,1,Normal,8000,15.360,1604.0,7269.0,1.604,7.269,5.665,False,False,False
520,63658468_7.2_0_p2_3629,0,Normal,8000,15.360,6471.0,11438.0,6.471,11.438,4.967,False,False,False
830,65038439_5.7_1_p3_3462,1,Normal,8000,15.360,4782.0,9204.0,4.782,9.204,4.422,False,False,False
543,63658468_7.2_0_p4_3630,0,Normal,8000,15.360,9320.0,13621.0,9.320,13.621,4.301,False,False,False
829,65038439_5.7_1_p3_3462,0,Normal,8000,15.360,10383.0,14668.0,10.383,14.668,4.285,False,False,False
102,41031554_10.7_0_p1_4067,0,Normal,8000,9.216,1771.0,5981.0,1.771,5.981,4.210,False,False,False
1836,41260853_9.7_1_p2_574,1,Normal,8000,15.360,10781.0,14936.0,10.781,14.936,4.155,False,False,False
1112,65107666_9.6_0_p4_3532,0,Normal,8000,9.216,2740.0,6780.0,2.740,6.780,4.040,False,False,False
1111,65107666_9.6_0_p4_3527,2,Normal,8000,9.216,1247.0,5222.0,1.247,5.222,3.975,False,False,False
1995,63723123_8.9_0_p1_2091,1,Normal,8000,15.360,3566.0,7536.0,3.566,7.536,3.970,False,False,False


In [9]:
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
import torch.nn.functional as F

# =========================================================
# ICBHI FINAL PIPELINE PARAMETERS
# =========================================================

TARGET_SR = 4000
TARGET_SEC = 4.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_SEC)   # 16000

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 128
F_MIN = 0
F_MAX = 2000

DB_MIN = -80.0
DB_MAX = 0.0


# =========================================================
# Mel transform
# =========================================================

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=F_MIN,
    f_max=F_MAX,
    power=2.0
)


# =========================================================
# 1. SPRSound event 추출
# =========================================================

def load_sprsound_event(row):

    wav, sr = sf.read(row["wav_path"], dtype="float32")

    # stereo라면 mono
    if wav.ndim > 1:
        wav = wav.mean(axis=1)

    start_sec = float(row["start_raw"]) / 1000.0
    end_sec = float(row["end_raw"]) / 1000.0

    start_sample = int(round(start_sec * sr))
    end_sample = int(round(end_sec * sr))

    event = wav[start_sample:end_sample]

    return torch.tensor(event, dtype=torch.float32), sr


# =========================================================
# 2. 4 kHz resampling
# =========================================================

def resample_audio(waveform, orig_sr):

    if orig_sr != TARGET_SR:
        waveform = torchaudio.functional.resample(
            waveform,
            orig_freq=orig_sr,
            new_freq=TARGET_SR
        )

    return waveform


# =========================================================
# 3. 4초 표준화
#    짧으면 symmetric zero-padding
#    길면 center crop
# =========================================================

def fix_length_4sec(waveform):

    n = waveform.numel()

    if n < TARGET_SAMPLES:

        total_pad = TARGET_SAMPLES - n

        left_pad = total_pad // 2
        right_pad = total_pad - left_pad

        waveform = F.pad(
            waveform,
            (left_pad, right_pad),
            mode="constant",
            value=0
        )

    elif n > TARGET_SAMPLES:

        start = (n - TARGET_SAMPLES) // 2
        waveform = waveform[start:start + TARGET_SAMPLES]

    return waveform


# =========================================================
# 4. Mel-spectrogram
# =========================================================

def make_mel(waveform):

    mel = mel_transform(waveform.unsqueeze(0))

    # power → dB
    mel_db = 10.0 * torch.log10(
        torch.clamp(mel, min=1e-10)
    )

    # sample별 최대값을 0 dB로
    mel_db = mel_db - mel_db.amax(
        dim=(-2, -1),
        keepdim=True
    )

    # -80 ~ 0 dB
    mel_db = torch.clamp(
        mel_db,
        min=DB_MIN,
        max=DB_MAX
    )

    # 0 ~ 1
    mel_norm = (mel_db - DB_MIN) / (DB_MAX - DB_MIN)

    return mel_norm


# =========================================================
# 5. AlexNet 입력
# =========================================================

def make_alexnet_input(mel):

    # 1 × 128 × T → 1 × 224 × 224
    image = F.interpolate(
        mel.unsqueeze(0),
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    ).squeeze(0)

    # 1 channel → 3 channel
    image = image.repeat(3, 1, 1)

    return image


# =========================================================
# 6. 전체 preprocessing
# =========================================================

def preprocess_sprsound(row):

    waveform, orig_sr = load_sprsound_event(row)

    original_samples = waveform.numel()
    original_duration = original_samples / orig_sr

    waveform = resample_audio(waveform, orig_sr)

    resampled_samples = waveform.numel()

    waveform = fix_length_4sec(waveform)

    mel = make_mel(waveform)

    image = make_alexnet_input(mel)

    return {
        "original_sr": orig_sr,
        "original_duration": original_duration,
        "original_samples": original_samples,
        "resampled_samples_before_fix": resampled_samples,
        "final_samples": waveform.numel(),
        "mel_shape": tuple(mel.shape),
        "alexnet_shape": tuple(image.shape),
        "mel_min": float(mel.min()),
        "mel_max": float(mel.max()),
    }


# =========================================================
# 대표 sample 선택
# =========================================================

primary_df = df[df["include_primary"] == True].copy()

samples = []

for label in ["Normal", "Wheeze", "Crackle", "Both"]:

    temp = primary_df[
        primary_df["event_label_icbhi"] == label
    ]

    if len(temp) > 0:
        samples.append(temp.iloc[0])

sample_df = pd.DataFrame(samples)


# =========================================================
# 실행
# =========================================================

results = []

for _, row in sample_df.iterrows():

    r = preprocess_sprsound(row)

    r["record_id"] = row["record_id"]
    r["original_label"] = row["event_label_original"]
    r["icbhi_label"] = row["event_label_icbhi"]

    results.append(r)


result_df = pd.DataFrame(results)

display(result_df)

print("\n" + "=" * 80)
print("Preprocessing sanity check")
print("=" * 80)

print("\n최종 sample 수:")
print(result_df["final_samples"].value_counts())

print("\nMel shape:")
print(result_df["mel_shape"].value_counts())

print("\nAlexNet input shape:")
print(result_df["alexnet_shape"].value_counts())

print("\nMel value 범위:")
print(
    result_df[["mel_min", "mel_max"]]
)

,original_sr,original_duration,original_samples,resampled_samples_before_fix,final_samples,mel_shape,alexnet_shape,mel_min,mel_max,record_id,original_label,icbhi_label
0,8000,1.606,12848,6424,16000,"(1, 128, 126)","(3, 224, 224)",0.0,1.0,40512331_8.1_1_p1_3548,Normal,Normal
1,8000,1.107,8856,4428,16000,"(1, 128, 126)","(3, 224, 224)",0.0,1.0,41092434_4.8_0_p1_3493,Wheeze,Wheeze
2,8000,1.348,10784,5392,16000,"(1, 128, 126)","(3, 224, 224)",0.0,1.0,41225759_7.2_1_p2_4211,Fine Crackle,Crackle
3,8000,1.897,15176,7588,16000,"(1, 128, 126)","(3, 224, 224)",0.0,1.0,41225759_7.2_1_p2_4202,Wheeze+Crackle,Both



Preprocessing sanity check

최종 sample 수:
final_samples
16000    4
Name: count, dtype: int64

Mel shape:
mel_shape
(1, 128, 126)    4
Name: count, dtype: int64

AlexNet input shape:
alexnet_shape
(3, 224, 224)    4
Name: count, dtype: int64

Mel value 범위:
   mel_min  mel_max
0      0.0      1.0
1      0.0      1.0
2      0.0      1.0
3      0.0      1.0


In [10]:
import numpy as np
import pandas as pd
import torch

print("=" * 80)
print("SPRSound 전체 Primary Event Preprocessing Audit")
print("=" * 80)

primary_df = df[df["include_primary"] == True].copy()

audit_rows = []

for i, (_, row) in enumerate(primary_df.iterrows(), start=1):

    try:
        waveform, orig_sr = load_sprsound_event(row)

        original_samples = waveform.numel()
        original_duration = original_samples / orig_sr

        # resampling
        waveform = resample_audio(waveform, orig_sr)
        resampled_samples = waveform.numel()

        # 4초 고정
        waveform = fix_length_4sec(waveform)

        # Mel
        mel = make_mel(waveform)

        # AlexNet input
        image = make_alexnet_input(mel)

        audit_rows.append({
            "record_id": row["record_id"],
            "event_id": row["event_id"],
            "label": row["event_label_icbhi"],

            "original_sr": orig_sr,
            "original_duration_sec": original_duration,
            "resampled_samples_before_fix": resampled_samples,

            "final_samples": waveform.numel(),
            "mel_shape": tuple(mel.shape),
            "alexnet_shape": tuple(image.shape),

            "mel_min": float(mel.min()),
            "mel_max": float(mel.max()),

            "mel_has_nan": bool(torch.isnan(mel).any()),
            "mel_has_inf": bool(torch.isinf(mel).any()),

            "image_has_nan": bool(torch.isnan(image).any()),
            "image_has_inf": bool(torch.isinf(image).any()),

            "success": True,
            "error": None,
        })

    except Exception as e:

        audit_rows.append({
            "record_id": row["record_id"],
            "event_id": row["event_id"],
            "label": row["event_label_icbhi"],

            "success": False,
            "error": str(e),
        })

    if i % 250 == 0:
        print(f"{i} / {len(primary_df)} 처리 완료")


audit_df = pd.DataFrame(audit_rows)


print("\n" + "=" * 80)
print("AUDIT RESULT")
print("=" * 80)

print("\n전체 primary events:")
print(len(audit_df))

print("\n성공 / 실패:")
print(audit_df["success"].value_counts())


successful = audit_df[audit_df["success"] == True].copy()


print("\nFinal samples:")
print(successful["final_samples"].value_counts())


print("\nMel shape:")
print(successful["mel_shape"].value_counts())


print("\nAlexNet shape:")
print(successful["alexnet_shape"].value_counts())


print("\nNaN / Inf:")
print("Mel NaN   :", successful["mel_has_nan"].sum())
print("Mel Inf   :", successful["mel_has_inf"].sum())
print("Image NaN :", successful["image_has_nan"].sum())
print("Image Inf :", successful["image_has_inf"].sum())


print("\nMel min 범위:")
print(
    successful["mel_min"].min(),
    "~",
    successful["mel_min"].max()
)

print("\nMel max 범위:")
print(
    successful["mel_max"].min(),
    "~",
    successful["mel_max"].max()
)


print("\nClass별 처리 수:")
print(successful["label"].value_counts())


# 실패 데이터가 있으면 표시
failed = audit_df[audit_df["success"] == False]

if len(failed) > 0:
    print("\n[실패 event]")
    display(failed)
else:
    print("\n실패 event 없음")

SPRSound 전체 Primary Event Preprocessing Audit
250 / 2417 처리 완료
500 / 2417 처리 완료
750 / 2417 처리 완료
1000 / 2417 처리 완료
1250 / 2417 처리 완료
1500 / 2417 처리 완료
1750 / 2417 처리 완료
2000 / 2417 처리 완료
2250 / 2417 처리 완료

AUDIT RESULT

전체 primary events:
2417

성공 / 실패:
success
True    2417
Name: count, dtype: int64

Final samples:
final_samples
16000    2417
Name: count, dtype: int64

Mel shape:
mel_shape
(1, 128, 126)    2417
Name: count, dtype: int64

AlexNet shape:
alexnet_shape
(3, 224, 224)    2417
Name: count, dtype: int64

NaN / Inf:
Mel NaN   : 0
Mel Inf   : 0
Image NaN : 0
Image Inf : 0

Mel min 범위:
0.0 ~ 0.05837411805987358

Mel max 범위:
1.0 ~ 1.0

Class별 처리 수:
label
Normal     1728
Wheeze      413
Crackle     272
Both          4
Name: count, dtype: int64

실패 event 없음


In [11]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\2026\final_project")

print("=" * 80)
print("ICBHI Internal Test 평가용 파일 탐색")
print("=" * 80)

# ---------------------------------------------------------
# 1. 모델 파일
# ---------------------------------------------------------

model_files = []

for ext in ["*.pth", "*.pt", "*.ckpt"]:
    model_files.extend(PROJECT_ROOT.rglob(ext))

print("\n[모델 파일]")
print("-" * 80)

if model_files:
    for p in sorted(model_files):
        print(p)
else:
    print("모델 파일 없음")


# ---------------------------------------------------------
# 2. CSV / manifest / split 관련 파일
# ---------------------------------------------------------

candidate_files = []

keywords = [
    "test",
    "split",
    "manifest",
    "icbhi",
    "alexnet",
]

for p in PROJECT_ROOT.rglob("*.csv"):

    name_lower = p.name.lower()

    if any(k in name_lower for k in keywords):
        candidate_files.append(p)


print("\n" + "=" * 80)
print("[ICBHI / Test / Split 관련 CSV]")
print("-" * 80)

if candidate_files:
    for p in sorted(candidate_files):
        print(p)
else:
    print("관련 CSV 없음")


# ---------------------------------------------------------
# 3. AlexNet 관련 파일 전체
# ---------------------------------------------------------

alexnet_files = []

for p in PROJECT_ROOT.rglob("*"):

    if p.is_file() and "alexnet" in p.name.lower():
        alexnet_files.append(p)


print("\n" + "=" * 80)
print("[AlexNet 관련 파일]")
print("-" * 80)

if alexnet_files:
    for p in sorted(alexnet_files):
        print(p)
else:
    print("AlexNet 이름이 포함된 파일 없음")

ICBHI Internal Test 평가용 파일 탐색

[모델 파일]
--------------------------------------------------------------------------------
D:\2026\final_project\backup\respiratory_project_backup\results\trackB\checkpoints\AlexNet_Base_Final_Selected_seed777.pth
D:\2026\final_project\backup\respiratory_project_backup\results\trackB\checkpoints\ResNet50_Stage1_Extended_seed777.pth
D:\2026\final_project\backup\respiratory_project_backup\results\trackB\checkpoints\ResNet50_Stage2B_WD1e-3_seed777.pth
D:\2026\final_project\results\checkpoints\AlexNet_Base_Final_Selected_seed777.pth
D:\2026\final_project\results\checkpoints\ResNet50_Stage1_Extended_seed777.pth
D:\2026\final_project\results\checkpoints\ResNet50_Stage2B_WD1e-3_seed777.pth

[ICBHI / Test / Split 관련 CSV]
--------------------------------------------------------------------------------
D:\2026\final_project\backup\respiratory_project_backup\metadata\ICBHI_cycles_4k_4sec_metadata.csv
D:\2026\final_project\backup\respiratory_project_backup\metadata\ICB

In [12]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"D:\2026\final_project")

TEST_RESULT = (
    PROJECT_ROOT
    / "results"
    / "AlexNet_Final_Independent_Test_seed777.csv"
)

VAL_TEST_RESULT = (
    PROJECT_ROOT
    / "results"
    / "AlexNet_Validation_vs_Test_seed777.csv"
)

CHECKPOINT = (
    PROJECT_ROOT
    / "results"
    / "checkpoints"
    / "AlexNet_Base_Final_Selected_seed777.pth"
)

print("=" * 80)
print("ICBHI 최종 AlexNet 기존 결과 확인")
print("=" * 80)

print("\n[Checkpoint]")
print(CHECKPOINT)
print("존재:", CHECKPOINT.exists())

print("\n[Independent Test CSV]")
print(TEST_RESULT)
print("존재:", TEST_RESULT.exists())

print("\n[Validation vs Test CSV]")
print(VAL_TEST_RESULT)
print("존재:", VAL_TEST_RESULT.exists())


# =========================================================
# Independent Test
# =========================================================

if TEST_RESULT.exists():

    test_df = pd.read_csv(TEST_RESULT)

    print("\n" + "=" * 80)
    print("Independent Test CSV")
    print("=" * 80)

    print("\nShape:")
    print(test_df.shape)

    print("\nColumns:")
    print(test_df.columns.tolist())

    print("\n전체 내용:")
    display(test_df)


# =========================================================
# Validation vs Test
# =========================================================

if VAL_TEST_RESULT.exists():

    val_test_df = pd.read_csv(VAL_TEST_RESULT)

    print("\n" + "=" * 80)
    print("Validation vs Test CSV")
    print("=" * 80)

    print("\nShape:")
    print(val_test_df.shape)

    print("\nColumns:")
    print(val_test_df.columns.tolist())

    print("\n전체 내용:")
    display(val_test_df)

ICBHI 최종 AlexNet 기존 결과 확인

[Checkpoint]
D:\2026\final_project\results\checkpoints\AlexNet_Base_Final_Selected_seed777.pth
존재: True

[Independent Test CSV]
D:\2026\final_project\results\AlexNet_Final_Independent_Test_seed777.csv
존재: True

[Validation vs Test CSV]
D:\2026\final_project\results\AlexNet_Validation_vs_Test_seed777.csv
존재: True

Independent Test CSV

Shape:
(1, 7)

Columns:
['Model', 'Split', 'N', 'Accuracy', 'Balanced_Accuracy', 'Macro_F1', 'Weighted_F1']

전체 내용:


,Model,Split,N,Accuracy,Balanced_Accuracy,Macro_F1,Weighted_F1
0,AlexNet Base,Independent Test,1018,0.527505,0.459836,0.424701,0.528638



Validation vs Test CSV

Shape:
(2, 5)

Columns:
['Dataset', 'Accuracy', 'Balanced_Accuracy', 'Macro_F1', 'Macro_F1_Difference_from_Validation']

전체 내용:


,Dataset,Accuracy,Balanced_Accuracy,Macro_F1,Macro_F1_Difference_from_Validation
0,Validation,0.566200,0.473600,0.456200,0.000000
1,Independent Test,0.527505,0.459836,0.424701,-0.031499


In [13]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\2026\final_project")

print("=" * 85)
print("ICBHI AlexNet 상세 Test 결과 파일 탐색")
print("=" * 85)

keywords = [
    "alexnet",
    "test",
    "prediction",
    "pred",
    "confusion",
    "classification",
    "report",
    "metric",
    "seed777",
]

extensions = {
    ".csv", ".xlsx", ".json", ".txt",
    ".npy", ".npz", ".pkl", ".joblib"
}

found = []

for p in PROJECT_ROOT.rglob("*"):

    if not p.is_file():
        continue

    # backup 폴더는 제외
    if "backup" in [part.lower() for part in p.parts]:
        continue

    if p.suffix.lower() not in extensions:
        continue

    name = p.name.lower()

    score = sum(k in name for k in keywords)

    if score >= 2:
        found.append((score, p))


found = sorted(
    found,
    key=lambda x: (-x[0], str(x[1]).lower())
)

print(f"\n후보 파일 수: {len(found)}\n")

for score, p in found:
    print(f"[score={score}] {p}")

ICBHI AlexNet 상세 Test 결과 파일 탐색

후보 파일 수: 4

[score=3] D:\2026\final_project\results\AlexNet_Final_Independent_Test_seed777.csv
[score=3] D:\2026\final_project\results\AlexNet_Validation_vs_Test_seed777.csv
[score=2] D:\2026\final_project\external_data_processed\HF_Lung_V1\results\HF_Lung_V1_external_validation_classification_report.csv
[score=2] D:\2026\final_project\external_data_processed\HF_Lung_V1\results\HF_Lung_V1_external_validation_predictions.csv


In [14]:
print("\n" + "=" * 85)
print("Patient-level split 파일 확인")
print("=" * 85)

split_files = []

for p in PROJECT_ROOT.rglob("*.csv"):

    if "backup" in [part.lower() for part in p.parts]:
        continue

    name = p.name.lower()

    if "patient" in name and "split" in name:
        split_files.append(p)

if split_files:
    for p in sorted(split_files):
        print(p)
else:
    print("현재 본본 프로젝트에서 patient-level split CSV를 찾지 못했습니다.")


Patient-level split 파일 확인
D:\2026\final_project\data_processed\metadata\ICBHI_patient_level_split_seed777_config.csv
D:\2026\final_project\data_processed\metadata\ICBHI_patient_level_split_seed777_v1.csv
D:\2026\final_project\data_processed\metadata\ICBHI_patient_level_split_v1.csv


In [15]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"D:\2026\final_project")

META_DIR = PROJECT_ROOT / "data_processed" / "metadata"

SPLIT_CONFIG = META_DIR / "ICBHI_patient_level_split_seed777_config.csv"
SPLIT_FILE = META_DIR / "ICBHI_patient_level_split_seed777_v1.csv"
CYCLE_META = META_DIR / "ICBHI_cycles_4k_4sec_metadata.csv"

print("=" * 85)
print("ICBHI seed777 split / metadata 확인")
print("=" * 85)


# =========================================================
# 1. split config
# =========================================================

print("\n[1] Split config")
print("-" * 85)

print("경로:", SPLIT_CONFIG)
print("존재:", SPLIT_CONFIG.exists())

if SPLIT_CONFIG.exists():
    split_config_df = pd.read_csv(SPLIT_CONFIG)

    print("\nShape:")
    print(split_config_df.shape)

    print("\nColumns:")
    print(split_config_df.columns.tolist())

    print("\n내용:")
    display(split_config_df)


# =========================================================
# 2. patient-level split
# =========================================================

print("\n" + "=" * 85)
print("[2] Patient-level split")
print("-" * 85)

print("경로:", SPLIT_FILE)
print("존재:", SPLIT_FILE.exists())

if SPLIT_FILE.exists():
    split_df = pd.read_csv(SPLIT_FILE)

    print("\nShape:")
    print(split_df.shape)

    print("\nColumns:")
    print(split_df.columns.tolist())

    print("\n앞 10개:")
    display(split_df.head(10))

    print("\n각 column unique 개수:")
    for col in split_df.columns:
        print(f"{col:30s}: {split_df[col].nunique(dropna=False)}")


# =========================================================
# 3. cycle metadata
# =========================================================

print("\n" + "=" * 85)
print("[3] ICBHI cycle metadata")
print("-" * 85)

print("경로:", CYCLE_META)
print("존재:", CYCLE_META.exists())

if CYCLE_META.exists():
    cycle_df = pd.read_csv(CYCLE_META)

    print("\nShape:")
    print(cycle_df.shape)

    print("\nColumns:")
    print(cycle_df.columns.tolist())

    print("\n앞 10개:")
    display(cycle_df.head(10))

ICBHI seed777 split / metadata 확인

[1] Split config
-------------------------------------------------------------------------------------
경로: D:\2026\final_project\data_processed\metadata\ICBHI_patient_level_split_seed777_config.csv
존재: True

Shape:
(1, 10)

Columns:
['split_version', 'seed', 'train_patients', 'validation_patients', 'test_patients', 'train_cycles', 'validation_cycles', 'test_cycles', 'split_score', 'patient_overlap']

내용:


,split_version,seed,train_patients,validation_patients,test_patients,train_cycles,validation_cycles,test_cycles,split_score,patient_overlap
0,v1,777,88,19,19,4737,1141,1018,0.158224,0



[2] Patient-level split
-------------------------------------------------------------------------------------
경로: D:\2026\final_project\data_processed\metadata\ICBHI_patient_level_split_seed777_v1.csv
존재: True

Shape:
(6896, 21)

Columns:
['patient_id', 'source_wav', 'cycle_id', 'start', 'end', 'original_duration', 'crackle', 'wheeze', 'label', 'original_sr', 'target_sr', 'normalization_method', 'processed_samples', 'processed_duration', 'processed_file', 'use_for_model', 'processed_name', 'near_fullscale_ratio', 'rms', 'quality_flag', 'split']

앞 10개:


,patient_id,source_wav,cycle_id,start,end,original_duration,crackle,wheeze,label,original_sr,...,normalization_method,processed_samples,processed_duration,processed_file,use_for_model,processed_name,near_fullscale_ratio,rms,quality_flag,split
0,101,101_1b1_Al_sc_Meditron.wav,0,0.036,0.579,0.543,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle00_Normal.wav,0.0,0.029413,OK,train
1,101,101_1b1_Al_sc_Meditron.wav,1,0.579,2.450,1.871,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle01_Normal.wav,0.0,0.050298,OK,train
2,101,101_1b1_Al_sc_Meditron.wav,2,2.450,3.893,1.443,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle02_Normal.wav,0.0,0.042131,OK,train
3,101,101_1b1_Al_sc_Meditron.wav,3,3.893,5.793,1.900,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle03_Normal.wav,0.0,0.044338,OK,train
4,101,101_1b1_Al_sc_Meditron.wav,4,5.793,7.521,1.728,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle04_Normal.wav,0.0,0.037884,OK,train
5,101,101_1b1_Al_sc_Meditron.wav,5,7.521,9.279,1.758,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle05_Normal.wav,0.0,0.036922,OK,train
6,101,101_1b1_Al_sc_Meditron.wav,6,9.279,11.150,1.871,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle06_Normal.wav,0.0,0.039681,OK,train
7,101,101_1b1_Al_sc_Meditron.wav,7,11.150,13.036,1.886,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle07_Normal.wav,0.0,0.035533,OK,train
8,101,101_1b1_Al_sc_Meditron.wav,8,13.036,14.721,1.685,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle08_Normal.wav,0.0,0.031531,OK,train
9,101,101_1b1_Al_sc_Meditron.wav,9,14.721,16.707,1.986,0,0,Normal,44100,...,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...,True,101_1b1_Al_sc_Meditron_cycle09_Normal.wav,0.0,0.041491,OK,train



각 column unique 개수:
patient_id                    : 126
source_wav                    : 920
cycle_id                      : 33
start                         : 2835
end                           : 2818
original_duration             : 2730
crackle                       : 2
wheeze                        : 2
label                         : 4
original_sr                   : 3
target_sr                     : 1
normalization_method          : 3
processed_samples             : 1
processed_duration            : 1
processed_file                : 6896
use_for_model                 : 1
processed_name                : 6896
near_fullscale_ratio          : 789
rms                           : 6896
quality_flag                  : 2
split                         : 3

[3] ICBHI cycle metadata
-------------------------------------------------------------------------------------
경로: D:\2026\final_project\data_processed\metadata\ICBHI_cycles_4k_4sec_metadata.csv
존재: True

Shape:
(6898, 15)

Columns:
['pati

,patient_id,source_wav,cycle_id,start,end,original_duration,crackle,wheeze,label,original_sr,target_sr,normalization_method,processed_samples,processed_duration,processed_file
0,101,101_1b1_Al_sc_Meditron.wav,0,0.036,0.579,0.543,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...
1,101,101_1b1_Al_sc_Meditron.wav,1,0.579,2.450,1.871,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...
2,101,101_1b1_Al_sc_Meditron.wav,2,2.450,3.893,1.443,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...
3,101,101_1b1_Al_sc_Meditron.wav,3,3.893,5.793,1.900,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...
4,101,101_1b1_Al_sc_Meditron.wav,4,5.793,7.521,1.728,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...
5,101,101_1b1_Al_sc_Meditron.wav,5,7.521,9.279,1.758,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...
6,101,101_1b1_Al_sc_Meditron.wav,6,9.279,11.150,1.871,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...
7,101,101_1b1_Al_sc_Meditron.wav,7,11.150,13.036,1.886,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...
8,101,101_1b1_Al_sc_Meditron.wav,8,13.036,14.721,1.685,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...
9,101,101_1b1_Al_sc_Meditron.wav,9,14.721,16.707,1.986,0,0,Normal,44100,4000,padding,16000,4.0,D:\2026\final_project\data_processed\cycles_4k...


In [16]:
from pathlib import Path
import pandas as pd

SPLIT_FILE = (
    Path(r"D:\2026\final_project\data_processed\metadata")
    / "ICBHI_patient_level_split_seed777_v1.csv"
)

split_df = pd.read_csv(SPLIT_FILE)

print("=" * 85)
print("ICBHI seed777 Independent Test 최종 Audit")
print("=" * 85)

# =========================================================
# 전체 split 분포
# =========================================================

print("\n[1] Cycle 수 by split")
print(split_df["split"].value_counts())

print("\n[2] Patient 수 by split")
print(
    split_df.groupby("split")["patient_id"]
    .nunique()
)

# =========================================================
# Test subset
# =========================================================

test_df = split_df[
    split_df["split"].astype(str).str.lower() == "test"
].copy()

print("\n" + "=" * 85)
print("[3] Independent Test")
print("=" * 85)

print("Test cycles   :", len(test_df))
print("Test patients :", test_df["patient_id"].nunique())

print("\nClass 분포:")
print(test_df["label"].value_counts())

print("\nClass 비율:")
print(
    (
        test_df["label"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
)

# =========================================================
# use_for_model / quality
# =========================================================

print("\n[4] use_for_model")
print(test_df["use_for_model"].value_counts(dropna=False))

print("\n[5] quality_flag")
print(test_df["quality_flag"].value_counts(dropna=False))

# =========================================================
# processed file 존재 확인
# =========================================================

test_df["processed_file_exists"] = (
    test_df["processed_file"]
    .astype(str)
    .apply(lambda x: Path(x).exists())
)

print("\n[6] Processed WAV 존재 여부")
print(test_df["processed_file_exists"].value_counts())

# =========================================================
# 환자 overlap 직접 재검증
# =========================================================

train_patients = set(
    split_df.loc[
        split_df["split"].astype(str).str.lower() == "train",
        "patient_id"
    ]
)

val_patients = set(
    split_df.loc[
        split_df["split"].astype(str).str.lower().isin(
            ["validation", "val"]
        ),
        "patient_id"
    ]
)

test_patients = set(test_df["patient_id"])

print("\n[7] Patient overlap")
print("Train ∩ Validation :", len(train_patients & val_patients))
print("Train ∩ Test       :", len(train_patients & test_patients))
print("Validation ∩ Test  :", len(val_patients & test_patients))

ICBHI seed777 Independent Test 최종 Audit

[1] Cycle 수 by split
split
train         4737
validation    1141
test          1018
Name: count, dtype: int64

[2] Patient 수 by split
split
test          19
train         88
validation    19
Name: patient_id, dtype: int64

[3] Independent Test
Test cycles   : 1018
Test patients : 19

Class 분포:
label
Normal     513
Crackle    269
Wheeze     159
Both        77
Name: count, dtype: int64

Class 비율:
label
Normal     50.39
Crackle    26.42
Wheeze     15.62
Both        7.56
Name: proportion, dtype: float64

[4] use_for_model
use_for_model
True    1018
Name: count, dtype: int64

[5] quality_flag
quality_flag
OK       995
CHECK     23
Name: count, dtype: int64

[6] Processed WAV 존재 여부
processed_file_exists
True    1018
Name: count, dtype: int64

[7] Patient overlap
Train ∩ Validation : 0
Train ∩ Test       : 0
Validation ∩ Test  : 0


In [17]:
print("=" * 80)
print("ICBHI Test 누락 항목 확인")
print("=" * 80)

print("\n[1] Test class 분포")
print(test_df["label"].value_counts())

print("\n[2] Test class 비율 (%)")
print(
    test_df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\n[3] use_for_model")
print(test_df["use_for_model"].value_counts(dropna=False))

print("\n[4] quality_flag")
print(test_df["quality_flag"].value_counts(dropna=False))


ICBHI Test 누락 항목 확인

[1] Test class 분포
label
Normal     513
Crackle    269
Wheeze     159
Both        77
Name: count, dtype: int64

[2] Test class 비율 (%)
label
Normal     50.39
Crackle    26.42
Wheeze     15.62
Both        7.56
Name: proportion, dtype: float64

[3] use_for_model
use_for_model
True    1018
Name: count, dtype: int64

[4] quality_flag
quality_flag
OK       995
CHECK     23
Name: count, dtype: int64


In [18]:
print("=" * 70)
print("ICBHI Test processed file 최종 확인")
print("=" * 70)

print(
    test_df["processed_file"]
    .astype(str)
    .apply(lambda x: Path(x).exists())
    .value_counts(dropna=False)
)

ICBHI Test processed file 최종 확인
processed_file
True    1018
Name: count, dtype: int64


In [19]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torchvision.models import alexnet

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 0. 경로
# ============================================================

PROJECT_ROOT = Path(r"D:\2026\final_project")

CHECKPOINT = (
    PROJECT_ROOT
    / "results"
    / "checkpoints"
    / "AlexNet_Base_Final_Selected_seed777.pth"
)

SPLIT_FILE = (
    PROJECT_ROOT
    / "data_processed"
    / "metadata"
    / "ICBHI_patient_level_split_seed777_v1.csv"
)

OUT_DIR = PROJECT_ROOT / "results" / "ICBHI_internal_test_detailed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRED_CSV = OUT_DIR / "AlexNet_ICBHI_Test_predictions_seed777.csv"
REPORT_CSV = OUT_DIR / "AlexNet_ICBHI_Test_classification_report_seed777.csv"
CM_CSV = OUT_DIR / "AlexNet_ICBHI_Test_confusion_matrix_seed777.csv"
SUMMARY_CSV = OUT_DIR / "AlexNet_ICBHI_Test_summary_seed777.csv"


# ============================================================
# 1. 재현 설정
# ============================================================

SEED = 777

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 85)
print("ICBHI AlexNet Independent Test 재평가")
print("=" * 85)

print("\nDevice     :", device)
print("Checkpoint :", CHECKPOINT)
print("Split      :", SPLIT_FILE)


# ============================================================
# 2. Test metadata
# ============================================================

split_df = pd.read_csv(SPLIT_FILE)

test_df = split_df[
    split_df["split"].astype(str).str.lower() == "test"
].copy()

test_df = test_df[
    test_df["use_for_model"] == True
].reset_index(drop=True)

print("\nTest cycles   :", len(test_df))
print("Test patients :", test_df["patient_id"].nunique())

print("\nTrue class distribution:")
print(test_df["label"].value_counts())


# ============================================================
# 3. 클래스 순서
# ============================================================

CLASS_NAMES = [
    "Normal",
    "Crackle",
    "Wheeze",
    "Both"
]

CLASS_TO_IDX = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

IDX_TO_CLASS = {
    i: name for name, i in CLASS_TO_IDX.items()
}


# ============================================================
# 4. ICBHI Final preprocessing
# ============================================================

TARGET_SR = 4000
TARGET_SAMPLES = 16000

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 128
F_MIN = 0
F_MAX = 2000

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=F_MIN,
    f_max=F_MAX,
    power=2.0
)


def preprocess_icbhi_wav(wav_path):

    wav, sr = sf.read(
        str(wav_path),
        dtype="float32"
    )

    # Stereo → mono
    if wav.ndim > 1:
        wav = wav.mean(axis=1)

    waveform = torch.tensor(
        wav,
        dtype=torch.float32
    )

    # 혹시 metadata와 다르게 들어온 경우에만 resample
    if sr != TARGET_SR:
        waveform = torchaudio.functional.resample(
            waveform,
            orig_freq=sr,
            new_freq=TARGET_SR
        )

    # --------------------------------------------------------
    # 4초 고정
    # --------------------------------------------------------

    n = waveform.numel()

    if n < TARGET_SAMPLES:

        pad_total = TARGET_SAMPLES - n

        pad_left = pad_total // 2
        pad_right = pad_total - pad_left

        waveform = F.pad(
            waveform,
            (pad_left, pad_right),
            mode="constant",
            value=0
        )

    elif n > TARGET_SAMPLES:

        start = (n - TARGET_SAMPLES) // 2

        waveform = waveform[
            start:start + TARGET_SAMPLES
        ]

    # --------------------------------------------------------
    # Mel spectrogram
    # --------------------------------------------------------

    mel = mel_transform(
        waveform.unsqueeze(0)
    )

    # power → dB
    mel_db = 10.0 * torch.log10(
        torch.clamp(mel, min=1e-10)
    )

    # sample별 peak = 0 dB
    mel_db = mel_db - mel_db.amax(
        dim=(-2, -1),
        keepdim=True
    )

    # -80 ~ 0 dB
    mel_db = torch.clamp(
        mel_db,
        min=-80.0,
        max=0.0
    )

    # 0 ~ 1
    mel_norm = (
        mel_db + 80.0
    ) / 80.0

    # --------------------------------------------------------
    # AlexNet input
    # --------------------------------------------------------

    image = F.interpolate(
        mel_norm.unsqueeze(0),
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    ).squeeze(0)

    # 1 channel → RGB 3 channel
    image = image.repeat(3, 1, 1)

    return image


# ============================================================
# 5. AlexNet 구성
# ============================================================

model = alexnet(weights=None)

model.classifier[6] = nn.Linear(
    model.classifier[6].in_features,
    len(CLASS_NAMES)
)


# ============================================================
# 6. Checkpoint 구조 확인 + 로드
# ============================================================

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint object type:")
print(type(checkpoint))

if isinstance(checkpoint, dict):

    print("\nCheckpoint keys:")
    print(list(checkpoint.keys())[:30])

    # 흔한 저장 형태 대응
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    else:
        # state_dict 자체가 저장된 경우
        state_dict = checkpoint

else:
    raise TypeError(
        "예상하지 못한 checkpoint 형식입니다."
    )


# DataParallel 저장 대응
clean_state_dict = {}

for k, v in state_dict.items():

    if k.startswith("module."):
        k = k[len("module."):]

    clean_state_dict[k] = v


load_result = model.load_state_dict(
    clean_state_dict,
    strict=True
)

print("\nCheckpoint load:")
print(load_result)

model = model.to(device)
model.eval()


# ============================================================
# 7. Independent Test inference
# ============================================================

y_true = []
y_pred = []
prediction_rows = []

print("\n" + "=" * 85)
print("Inference 시작")
print("=" * 85)

with torch.inference_mode():

    for i, row in test_df.iterrows():

        image = preprocess_icbhi_wav(
            Path(row["processed_file"])
        )

        x = image.unsqueeze(0).to(device)

        logits = model(x)

        probs = torch.softmax(
            logits,
            dim=1
        )[0]

        pred_idx = int(
            torch.argmax(probs).item()
        )

        true_label = str(row["label"])
        pred_label = IDX_TO_CLASS[pred_idx]

        y_true.append(true_label)
        y_pred.append(pred_label)

        prediction_rows.append({
            "patient_id": row["patient_id"],
            "processed_name": row["processed_name"],
            "true_label": true_label,
            "pred_label": pred_label,

            "prob_Normal": float(probs[0].cpu()),
            "prob_Crackle": float(probs[1].cpu()),
            "prob_Wheeze": float(probs[2].cpu()),
            "prob_Both": float(probs[3].cpu()),

            "quality_flag": row["quality_flag"],
        })

        if (i + 1) % 100 == 0:
            print(
                f"{i + 1:4d} / {len(test_df)} 완료"
            )


# ============================================================
# 8. 전체 지표
# ============================================================

accuracy = accuracy_score(
    y_true,
    y_pred
)

balanced_acc = balanced_accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=CLASS_NAMES,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=CLASS_NAMES,
    average="weighted",
    zero_division=0
)


print("\n" + "=" * 85)
print("ICBHI INTERNAL TEST RESULT")
print("=" * 85)

print(f"N                  : {len(y_true)}")
print(f"Accuracy           : {accuracy:.6f}")
print(f"Balanced Accuracy  : {balanced_acc:.6f}")
print(f"Macro F1           : {macro_f1:.6f}")
print(f"Weighted F1        : {weighted_f1:.6f}")


# ============================================================
# 9. Classification Report
# ============================================================

report_dict = classification_report(
    y_true,
    y_pred,
    labels=CLASS_NAMES,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_df = (
    pd.DataFrame(report_dict)
    .T
)

print("\n[Classification Report]")
display(
    report_df.round(4)
)


# ============================================================
# 10. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=CLASS_NAMES
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True_{x}" for x in CLASS_NAMES],
    columns=[f"Pred_{x}" for x in CLASS_NAMES]
)

print("\n[Confusion Matrix]")
display(cm_df)


# ============================================================
# 11. 저장
# ============================================================

pred_df = pd.DataFrame(
    prediction_rows
)

summary_df = pd.DataFrame([{
    "Model": "AlexNet Base",
    "Split": "Independent Test",
    "N": len(y_true),
    "Accuracy": accuracy,
    "Balanced_Accuracy": balanced_acc,
    "Macro_F1": macro_f1,
    "Weighted_F1": weighted_f1,
}])

pred_df.to_csv(
    PRED_CSV,
    index=False,
    encoding="utf-8-sig"
)

report_df.to_csv(
    REPORT_CSV,
    encoding="utf-8-sig"
)

cm_df.to_csv(
    CM_CSV,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 85)
print("저장 완료")
print("=" * 85)

print(PRED_CSV)
print(REPORT_CSV)
print(CM_CSV)
print(SUMMARY_CSV)

ICBHI AlexNet Independent Test 재평가

Device     : cpu
Checkpoint : D:\2026\final_project\results\checkpoints\AlexNet_Base_Final_Selected_seed777.pth
Split      : D:\2026\final_project\data_processed\metadata\ICBHI_patient_level_split_seed777_v1.csv

Test cycles   : 1018
Test patients : 19

True class distribution:
label
Normal     513
Crackle    269
Wheeze     159
Both        77
Name: count, dtype: int64

Checkpoint object type:
<class 'collections.OrderedDict'>

Checkpoint keys:
['features.0.weight', 'features.0.bias', 'features.3.weight', 'features.3.bias', 'features.6.weight', 'features.6.bias', 'features.8.weight', 'features.8.bias', 'features.10.weight', 'features.10.bias', 'classifier.1.weight', 'classifier.1.bias', 'classifier.4.weight', 'classifier.4.bias', 'classifier.6.weight', 'classifier.6.bias']

Checkpoint load:
<All keys matched successfully>

Inference 시작
 100 / 1018 완료
 200 / 1018 완료
 300 / 1018 완료
 400 / 1018 완료
 500 / 1018 완료
 600 / 1018 완료
 700 / 1018 완료
 800 / 1018 

,precision,recall,f1-score,support
Normal,0.4865,0.0351,0.0655,513.000
Crackle,0.6323,0.5242,0.5732,269.000
Wheeze,0.0431,0.1132,0.0624,159.000
Both,0.1441,0.6364,0.2350,77.000
accuracy,0.2220,0.2220,0.2220,0.222
macro avg,0.3265,0.3272,0.2340,1018.000
weighted avg,0.4299,0.2220,0.2120,1018.000



[Confusion Matrix]


,Pred_Normal,Pred_Crackle,Pred_Wheeze,Pred_Both
True_Normal,18,55,277,163
True_Crackle,1,141,114,13
True_Wheeze,14,12,18,115
True_Both,4,15,9,49



저장 완료
D:\2026\final_project\results\ICBHI_internal_test_detailed\AlexNet_ICBHI_Test_predictions_seed777.csv
D:\2026\final_project\results\ICBHI_internal_test_detailed\AlexNet_ICBHI_Test_classification_report_seed777.csv
D:\2026\final_project\results\ICBHI_internal_test_detailed\AlexNet_ICBHI_Test_confusion_matrix_seed777.csv
D:\2026\final_project\results\ICBHI_internal_test_detailed\AlexNet_ICBHI_Test_summary_seed777.csv


In [20]:
# ============================================================
# 실제 모델 output index 순서
# ============================================================

MODEL_CLASS_NAMES = [
    "Both",
    "Crackle",
    "Normal",
    "Wheeze"
]

IDX_TO_CLASS = {
    i: name
    for i, name in enumerate(MODEL_CLASS_NAMES)
}

# 보고서/표시용 순서
REPORT_CLASS_NAMES = [
    "Normal",
    "Crackle",
    "Wheeze",
    "Both"
]

In [21]:
prediction_rows.append({
    "patient_id": row["patient_id"],
    "processed_name": row["processed_name"],
    "true_label": true_label,
    "pred_label": pred_label,

    "prob_Both": float(probs[0].cpu()),
    "prob_Crackle": float(probs[1].cpu()),
    "prob_Normal": float(probs[2].cpu()),
    "prob_Wheeze": float(probs[3].cpu()),

    "quality_flag": row["quality_flag"],
})

In [22]:
macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=REPORT_CLASS_NAMES,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=REPORT_CLASS_NAMES,
    average="weighted",
    zero_division=0
)

In [23]:
report_dict = classification_report(
    y_true,
    y_pred,
    labels=REPORT_CLASS_NAMES,
    target_names=REPORT_CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

In [24]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=REPORT_CLASS_NAMES
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True_{x}" for x in REPORT_CLASS_NAMES],
    columns=[f"Pred_{x}" for x in REPORT_CLASS_NAMES]
)

In [25]:
from pathlib import Path
import hashlib

p1 = Path(
    r"D:\2026\final_project\results\checkpoints"
    r"\AlexNet_Base_Final_Selected_seed777.pth"
)

p2 = Path(
    r"D:\2026\final_project\results\trackB\checkpoints"
    r"\AlexNet_Base_Final_Selected_seed777.pth"
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("현재 사용 파일 :", p1.exists())
print("공식 TrackB 파일:", p2.exists())

if p1.exists():
    print("\nresults/checkpoints SHA256:")
    print(sha256_file(p1))

if p2.exists():
    print("\nresults/trackB/checkpoints SHA256:")
    print(sha256_file(p2))

if p1.exists() and p2.exists():
    print("\n동일 파일 여부:")
    print(sha256_file(p1) == sha256_file(p2))

현재 사용 파일 : True
공식 TrackB 파일: False

results/checkpoints SHA256:
69a5d6ff8c756df1cae62a31e04bd04742aeece79e0d4d84f356799b19770380


In [26]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

PROJECT_ROOT = Path(r"D:\2026\final_project")

OLD_PRED = (
    PROJECT_ROOT
    / "results"
    / "ICBHI_internal_test_detailed"
    / "AlexNet_ICBHI_Test_predictions_seed777.csv"
)

OUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "ICBHI_internal_test_detailed"
)

# ============================================================
# 1. 이전 inference 결과 로드
# ============================================================

pred_df = pd.read_csv(OLD_PRED)

print("=" * 85)
print("ICBHI AlexNet class-index 교정")
print("=" * 85)

print("\n입력 prediction 수:", len(pred_df))


# ============================================================
# 2. 실제 checkpoint class index 순서로 probability 재매핑
#
# 실제 model index:
# 0 = Both
# 1 = Crackle
# 2 = Normal
# 3 = Wheeze
#
# 이전 코드가 잘못 붙인 column:
# prob_Normal  -> index 0
# prob_Crackle -> index 1
# prob_Wheeze  -> index 2
# prob_Both    -> index 3
# ============================================================

corrected = pred_df.copy()

corrected["prob_Both_correct"] = pred_df["prob_Normal"]
corrected["prob_Crackle_correct"] = pred_df["prob_Crackle"]
corrected["prob_Normal_correct"] = pred_df["prob_Wheeze"]
corrected["prob_Wheeze_correct"] = pred_df["prob_Both"]


MODEL_CLASS_ORDER = [
    "Both",
    "Crackle",
    "Normal",
    "Wheeze"
]

prob_matrix = corrected[
    [
        "prob_Both_correct",
        "prob_Crackle_correct",
        "prob_Normal_correct",
        "prob_Wheeze_correct"
    ]
].to_numpy()

pred_idx = np.argmax(prob_matrix, axis=1)

corrected["pred_label_correct"] = [
    MODEL_CLASS_ORDER[i]
    for i in pred_idx
]


# ============================================================
# 3. Metrics
# ============================================================

y_true = corrected["true_label"].astype(str)
y_pred = corrected["pred_label_correct"].astype(str)

REPORT_ORDER = [
    "Both",
    "Crackle",
    "Normal",
    "Wheeze"
]

accuracy = accuracy_score(y_true, y_pred)

balanced_acc = balanced_accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=REPORT_ORDER,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=REPORT_ORDER,
    average="weighted",
    zero_division=0
)


print("\n" + "=" * 85)
print("CORRECTED ICBHI INTERNAL TEST")
print("=" * 85)

print(f"N                  : {len(corrected)}")
print(f"Accuracy           : {accuracy:.6f}")
print(f"Balanced Accuracy  : {balanced_acc:.6f}")
print(f"Macro F1           : {macro_f1:.6f}")
print(f"Weighted F1        : {weighted_f1:.6f}")


# ============================================================
# 4. Classification Report
# ============================================================

report_dict = classification_report(
    y_true,
    y_pred,
    labels=REPORT_ORDER,
    target_names=REPORT_ORDER,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).T

print("\n[Classification Report]")
display(report_df.round(4))


# ============================================================
# 5. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=REPORT_ORDER
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True_{x}" for x in REPORT_ORDER],
    columns=[f"Pred_{x}" for x in REPORT_ORDER]
)

print("\n[Confusion Matrix]")
display(cm_df)


# ============================================================
# 6. 정리된 prediction 파일 생성
# ============================================================

final_pred_df = corrected[
    [
        "patient_id",
        "processed_name",
        "true_label",
        "pred_label_correct",
        "prob_Normal_correct",
        "prob_Crackle_correct",
        "prob_Wheeze_correct",
        "prob_Both_correct",
        "quality_flag"
    ]
].copy()

final_pred_df = final_pred_df.rename(
    columns={
        "pred_label_correct": "pred_label",
        "prob_Normal_correct": "prob_Normal",
        "prob_Crackle_correct": "prob_Crackle",
        "prob_Wheeze_correct": "prob_Wheeze",
        "prob_Both_correct": "prob_Both"
    }
)


# ============================================================
# 7. 저장
# ============================================================

FINAL_PRED = OUT_DIR / "AlexNet_ICBHI_Test_predictions_seed777_FINAL.csv"

FINAL_REPORT = (
    OUT_DIR
    / "AlexNet_ICBHI_Test_classification_report_seed777_FINAL.csv"
)

FINAL_CM = (
    OUT_DIR
    / "AlexNet_ICBHI_Test_confusion_matrix_seed777_FINAL.csv"
)

FINAL_SUMMARY = (
    OUT_DIR
    / "AlexNet_ICBHI_Test_summary_seed777_FINAL.csv"
)


summary_df = pd.DataFrame([{
    "Model": "AlexNet Base",
    "Split": "Independent Test",
    "Seed": 777,
    "N": len(corrected),
    "Accuracy": accuracy,
    "Balanced_Accuracy": balanced_acc,
    "Macro_F1": macro_f1,
    "Weighted_F1": weighted_f1,
    "Checkpoint_SHA256":
        "69a5d6ff8c756df1cae62a31e04bd04742aeeece79e0d4d84f356799b19770380"
}])


final_pred_df.to_csv(
    FINAL_PRED,
    index=False,
    encoding="utf-8-sig"
)

report_df.to_csv(
    FINAL_REPORT,
    encoding="utf-8-sig"
)

cm_df.to_csv(
    FINAL_CM,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    FINAL_SUMMARY,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 85)
print("FINAL 결과 저장 완료")
print("=" * 85)

print(FINAL_PRED)
print(FINAL_REPORT)
print(FINAL_CM)
print(FINAL_SUMMARY)

ICBHI AlexNet class-index 교정

입력 prediction 수: 1018

CORRECTED ICBHI INTERNAL TEST
N                  : 1018
Accuracy           : 0.527505
Balanced Accuracy  : 0.459836
Macro F1           : 0.424832
Weighted F1        : 0.528623

[Classification Report]


,precision,recall,f1-score,support
Both,0.1081,0.0519,0.0702,77.0000
Crackle,0.6323,0.5242,0.5732,269.0000
Normal,0.6627,0.5400,0.5951,513.0000
Wheeze,0.3382,0.7233,0.4609,159.0000
accuracy,0.5275,0.5275,0.5275,0.5275
macro avg,0.4353,0.4598,0.4248,1018.0000
weighted avg,0.5620,0.5275,0.5286,1018.0000



[Confusion Matrix]


,Pred_Both,Pred_Crackle,Pred_Normal,Pred_Wheeze
True_Both,4,15,9,49
True_Crackle,1,141,114,13
True_Normal,18,55,277,163
True_Wheeze,14,12,18,115



FINAL 결과 저장 완료
D:\2026\final_project\results\ICBHI_internal_test_detailed\AlexNet_ICBHI_Test_predictions_seed777_FINAL.csv
D:\2026\final_project\results\ICBHI_internal_test_detailed\AlexNet_ICBHI_Test_classification_report_seed777_FINAL.csv
D:\2026\final_project\results\ICBHI_internal_test_detailed\AlexNet_ICBHI_Test_confusion_matrix_seed777_FINAL.csv
D:\2026\final_project\results\ICBHI_internal_test_detailed\AlexNet_ICBHI_Test_summary_seed777_FINAL.csv


In [27]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\2026\final_project")

keywords = [
    "AlexNet",
    "CrossEntropyLoss",
    "optimizer",
    "Adam",
    "AdamW",
    "DataLoader",
    "MelSpectrogram",
    "Best epoch",
    "seed777",
]

extensions = {
    ".py",
    ".ipynb",
}

results = []

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in extensions:
        continue

    # backup 제외
    if "backup" in [x.lower() for x in path.parts]:
        continue

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    matched = [
        k for k in keywords
        if k.lower() in text.lower()
    ]

    if len(matched) >= 3:
        results.append(
            (len(matched), path, matched)
        )

results = sorted(
    results,
    key=lambda x: (-x[0], str(x[1]))
)

print("=" * 90)
print("AlexNet Base 원본 Training 코드 탐색")
print("=" * 90)

for score, path, matched in results:
    print("\n" + "-" * 90)
    print(f"[score={score}] {path}")
    print("matched:", matched)

AlexNet Base 원본 Training 코드 탐색

------------------------------------------------------------------------------------------
[score=9] D:\2026\final_project\notebooks\02_TrackB_Mel_CNN.ipynb
matched: ['AlexNet', 'CrossEntropyLoss', 'optimizer', 'Adam', 'AdamW', 'DataLoader', 'MelSpectrogram', 'Best epoch', 'seed777']

------------------------------------------------------------------------------------------
[score=3] D:\2026\final_project\notebooks\01_ICBHI_preprocessing.ipynb
matched: ['Adam', 'MelSpectrogram', 'seed777']

------------------------------------------------------------------------------------------
[score=3] D:\2026\final_project\notebooks\03_HF_Lung_external_validation.ipynb
matched: ['AlexNet', 'MelSpectrogram', 'seed777']


In [29]:
from pathlib import Path
import json

NOTEBOOK = Path(
    r"D:\2026\final_project\notebooks\02_TrackB_Mel_CNN.ipynb"
)

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "alexnet",
    "crossentropyloss",
    "optimizer",
    "adam",
    "adamw",
    "learning_rate",
    "lr=",
    "class_weight",
    "weight=",
    "batch_size",
    "epochs",
    "scheduler",
    "mel",
    "normalize",
    "transform",
    "seed",
    "best",
    "checkpoint",
    "class_names",
    "classes",
    "label_encoder",
]

print("=" * 90)
print("02 TrackB Mel CNN.ipynb 핵심 학습 설정 추출")
print("=" * 90)

for i, cell in enumerate(nb["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    source_lower = source.lower()

    matched = [
        k for k in keywords
        if k.lower() in source_lower
    ]

    if matched:
        print("\n" + "=" * 90)
        print(f"[CODE CELL {i}]")
        print("matched:", matched)
        print("-" * 90)
        print(source[:6000])

02 TrackB Mel CNN.ipynb 핵심 학습 설정 추출

[CODE CELL 1]
matched: ['mel', 'seed']
------------------------------------------------------------------------------------------
ROOT = Path(r"D:\2026\final_project")

SPLIT_PATH = (
    ROOT
    / "data_processed"
    / "metadata"
    / "ICBHI_patient_level_split_seed777_v1.csv"
)

# Mel-Spectrogram parameters
SAMPLE_RATE = 4000
TARGET_DURATION = 4.0
TARGET_SAMPLES = 16000

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 128
FMIN = 0
FMAX = 2000

# CNN common input size
IMAGE_SIZE = 224

print("Split file:", SPLIT_PATH.exists())
print("Sample rate:", SAMPLE_RATE)
print("Target duration:", TARGET_DURATION)
print("Mel bins:", N_MELS)
print("CNN image size:", IMAGE_SIZE)

[CODE CELL 3]
matched: ['class_names']
------------------------------------------------------------------------------------------
CLASS_NAMES = [
    "Both",
    "Crackle",
    "Normal",
    "Wheeze"
]

CLASS_TO_IDX = {
    name: idx
    for idx, name in enumerate(CLASS_NAMES)
}

IDX_TO_CLASS

In [31]:
from pathlib import Path
import json

NOTEBOOK = Path(
    r"D:\2026\final_project\notebooks\02_TrackB_Mel_CNN.ipynb"
)

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

# 우리가 지금 반드시 확인해야 할 항목
targets = {
    "CLASS ORDER": [
        "CLASS_NAMES",
        "class_names",
        "classes",
        "class_to_idx",
        "label_to_idx",
    ],
    "ALEXNET MODEL": [
        "alexnet(",
        "AlexNet_Weights",
        "classifier[6]",
        "model.classifier",
    ],
    "LOSS / CLASS WEIGHT": [
        "CrossEntropyLoss",
        "class_weight",
        "class_weights",
        "criterion",
    ],
    "OPTIMIZER": [
        "optimizer =",
        "Adam(",
        "AdamW(",
        "lr=",
        "weight_decay",
    ],
    "TRAINING CONTROL": [
        "BATCH_SIZE",
        "batch_size",
        "EPOCHS",
        "epochs",
        "scheduler",
    ],
    "CHECKPOINT / BEST MODEL": [
        "best_macro",
        "best_f1",
        "best_epoch",
        "torch.save",
        "checkpoint",
    ],
    "AUGMENTATION": [
        "SpecAug",
        "FrequencyMasking",
        "TimeMasking",
        "augment",
        "augmentation",
    ],
}

print("=" * 95)
print("AlexNet Base 핵심 학습 설정 정밀 추출")
print("=" * 95)

for section, patterns in targets.items():

    print("\n\n" + "#" * 95)
    print(f"# {section}")
    print("#" * 95)

    found = False

    for i, cell in enumerate(nb["cells"]):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        source_lower = source.lower()

        if any(p.lower() in source_lower for p in patterns):

            found = True

            print(f"\n[CODE CELL {i}]")
            print("-" * 95)
            print(source)

    if not found:
        print("\n해당 코드 없음")

AlexNet Base 핵심 학습 설정 정밀 추출


###############################################################################################
# CLASS ORDER
###############################################################################################

[CODE CELL 3]
-----------------------------------------------------------------------------------------------
CLASS_NAMES = [
    "Both",
    "Crackle",
    "Normal",
    "Wheeze"
]

CLASS_TO_IDX = {
    name: idx
    for idx, name in enumerate(CLASS_NAMES)
}

IDX_TO_CLASS = {
    idx: name
    for name, idx in CLASS_TO_IDX.items()
}

print(CLASS_TO_IDX)

[CODE CELL 6]
-----------------------------------------------------------------------------------------------
CLASS_NAMES = [
    "Both",
    "Crackle",
    "Normal",
    "Wheeze"
]

CLASS_TO_IDX = {
    name: idx
    for idx, name in enumerate(CLASS_NAMES)
}

IDX_TO_CLASS = {
    idx: name
    for name, idx in CLASS_TO_IDX.items()
}

print(CLASS_TO_IDX)

[CODE CELL 7]
---------------------------------

In [33]:
from pathlib import Path
import json

NOTEBOOK = Path(
    r"D:\2026\final_project\notebooks\02_TrackB_Mel_CNN.ipynb"
)

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

targets = [
    "alexnet(",
    "AlexNet_Weights",
    "classifier[6]",
    "CrossEntropyLoss",
    "class_weight",
    "class_weights",
    "criterion",
    "optimizer =",
    "Adam(",
    "AdamW(",
    "lr=",
    "weight_decay",
    "BATCH_SIZE",
    "batch_size",
    "EPOCHS",
    "epochs",
    "scheduler",
    "best_macro",
    "best_f1",
    "best_epoch",
    "torch.save",
    "FrequencyMasking",
    "TimeMasking",
    "SpecAug",
]

print("=" * 95)
print("AlexNet Base 나머지 핵심 학습 설정")
print("=" * 95)

for i, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    source_lower = source.lower()

    matched = [
        t for t in targets
        if t.lower() in source_lower
    ]

    if matched:
        print("\n" + "=" * 95)
        print(f"[CODE CELL {i}]")
        print("matched:", matched)
        print("-" * 95)

        # 너무 길면 앞부분만
        lines = source.splitlines()

        for j, line in enumerate(lines):
            if any(t.lower() in line.lower() for t in targets):

                start = max(0, j - 4)
                end = min(len(lines), j + 10)

                print("\n".join(lines[start:end]))
                print("\n" + "-" * 60)

AlexNet Base 나머지 핵심 학습 설정

[CODE CELL 10]
matched: ['BATCH_SIZE', 'batch_size']
-----------------------------------------------------------------------------------------------
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)


------------------------------------------------------------
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,

------------------------------------------------------------
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,

-----------------------------------------------

In [34]:
from pathlib import Path
import json
import re

NOTEBOOK = Path(
    r"D:\2026\final_project\notebooks\02_TrackB_Mel_CNN.ipynb"
)

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

patterns = {
    "MODEL": [
        r"alexnet\s*\(",
        r"AlexNet_Weights",
        r"classifier\s*\[\s*6\s*\]",
    ],

    "LOSS": [
        r"CrossEntropyLoss",
        r"class_weights?",
        r"criterion\s*=",
    ],

    "OPTIMIZER": [
        r"optimizer\s*=",
        r"AdamW?\s*\(",
        r"learning_rate",
        r"\blr\s*=",
        r"weight_decay",
    ],

    "TRAINING": [
        r"BATCH_SIZE\s*=",
        r"EPOCHS?\s*=",
        r"NUM_EPOCHS\s*=",
        r"scheduler",
    ],

    "BEST MODEL": [
        r"best_epoch",
        r"best_macro",
        r"best_f1",
        r"torch\.save",
    ],

    "AUGMENTATION": [
        r"SpecAug",
        r"FrequencyMasking",
        r"TimeMasking",
        r"augment",
    ],
}

print("=" * 90)
print("AlexNet Base 핵심 설정 - compact search")
print("=" * 90)

for category, regex_list in patterns.items():

    print(f"\n[{category}]")

    found = False

    for cell_idx, cell in enumerate(nb["cells"]):

        if cell.get("cell_type") != "code":
            continue

        lines = "".join(cell.get("source", [])).splitlines()

        for line_no, line in enumerate(lines):

            if any(
                re.search(pattern, line, flags=re.IGNORECASE)
                for pattern in regex_list
            ):
                print(
                    f"Cell {cell_idx:02d} | "
                    f"{line_no + 1:03d} | "
                    f"{line.strip()}"
                )
                found = True

    if not found:
        print("없음")

AlexNet Base 핵심 설정 - compact search

[MODEL]
Cell 28 | 004 | alexnet_model = models.alexnet(
Cell 28 | 005 | weights=models.AlexNet_Weights.IMAGENET1K_V1
Cell 28 | 009 | alexnet_model.classifier[6] = nn.Linear(
Cell 28 | 010 | alexnet_model.classifier[6].in_features,
Cell 36 | 009 | vgg16_model.classifier[6] = nn.Linear(
Cell 36 | 010 | vgg16_model.classifier[6].in_features,
Cell 44 | 008 | vgg19_model.classifier[6] = nn.Linear(
Cell 44 | 009 | vgg19_model.classifier[6].in_features,
Cell 75 | 001 | alexnet_v2 = models.alexnet(
Cell 75 | 002 | weights=models.AlexNet_Weights.IMAGENET1K_V1
Cell 75 | 005 | alexnet_v2.classifier[6] = nn.Linear(
Cell 75 | 006 | alexnet_v2.classifier[6].in_features,
Cell 84 | 001 | alexnet_v2b = models.alexnet(
Cell 84 | 002 | weights=models.AlexNet_Weights.IMAGENET1K_V1
Cell 84 | 005 | alexnet_v2b.classifier[6] = nn.Linear(
Cell 84 | 006 | alexnet_v2b.classifier[6].in_features,
Cell 88 | 001 | alexnet_v2c = models.alexnet(
Cell 88 | 002 | weights=models.Alex

In [35]:
from pathlib import Path
import json

NOTEBOOK = Path(
    r"D:\2026\final_project\notebooks\02_TrackB_Mel_CNN.ipynb"
)

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

search_terms = [
    "alexnet_model",
    "CrossEntropyLoss",
    "criterion",
    "optimizer",
    "Adam(",
    "AdamW(",
    "weight_decay",
    "num_epochs",
    "epochs",
    "best_epoch",
    "best_macro",
    "best_f1",
    "torch.save",
]

print("=" * 95)
print("AlexNet Base 학습 조건 최종 추출")
print("=" * 95)

for i, cell in enumerate(nb["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(
        term.lower() in source.lower()
        for term in search_terms
    ):
        print("\n" + "=" * 95)
        print(f"[CELL {i}]")
        print("=" * 95)

        # 너무 긴 셀은 AlexNet 관련 주변 줄만 표시
        lines = source.splitlines()

        hits = []

        for j, line in enumerate(lines):
            if any(
                term.lower() in line.lower()
                for term in search_terms
            ):
                hits.append(j)

        shown = set()

        for j in hits:
            start = max(0, j - 5)
            end = min(len(lines), j + 12)

            for k in range(start, end):
                if k not in shown:
                    print(f"{k+1:03d} | {lines[k]}")
                    shown.add(k)

            print("-" * 70)

AlexNet Base 학습 조건 최종 추출

[CELL 18]
001 | criterion = nn.CrossEntropyLoss(
002 |     weight=class_weights
003 | )
----------------------------------------------------------------------

[CELL 19]
031 |                 non_blocking=True
032 |             )
033 | 
034 |             logits = model(x)
035 | 
036 |             loss = criterion(
037 |                 logits,
038 |                 y
039 |             )
040 | 
041 |             running_loss += (
042 |                 loss.item()
043 |                 * x.size(0)
044 |             )
045 | 
046 |             preds = torch.argmax(
047 |                 logits,
----------------------------------------------------------------------

[CELL 20]
003 | def train_model(
004 |     model,
005 |     train_loader,
006 |     val_loader,
007 |     input_prepare_fn,
008 |     optimizer,
009 |     max_epochs=30,
010 |     patience=6
011 | ):
012 | 
013 |     best_state = None
014 |     best_macro_f1 = -1.0
015 |     best_epoch = 0
016 | 
017 | 

In [36]:
from pathlib import Path
import json

NOTEBOOK = Path(
    r"D:\2026\final_project\notebooks\02_TrackB_Mel_CNN.ipynb"
)

OUTPUT_TXT = Path(
    r"D:\2026\final_project\notebooks\02_TrackB_Mel_CNN_FULL_CODE.txt"
)

with open(NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

lines_out = []

for i, cell in enumerate(nb["cells"]):

    cell_type = cell.get("cell_type", "unknown")
    source = "".join(cell.get("source", []))

    lines_out.append("=" * 100)
    lines_out.append(f"CELL {i} | TYPE: {cell_type}")
    lines_out.append("=" * 100)

    if cell_type == "code":
        code_lines = source.splitlines()

        for line_no, line in enumerate(code_lines, start=1):
            lines_out.append(
                f"{line_no:04d} | {line}"
            )

    else:
        lines_out.append(source)

    lines_out.append("")


OUTPUT_TXT.write_text(
    "\n".join(lines_out),
    encoding="utf-8"
)

print("전체 notebook 코드 추출 완료")
print("저장 위치:")
print(OUTPUT_TXT)

print("\n총 cell 수:", len(nb["cells"]))

code_cells = [
    c for c in nb["cells"]
    if c.get("cell_type") == "code"
]

print("총 code cell 수:", len(code_cells))

전체 notebook 코드 추출 완료
저장 위치:
D:\2026\final_project\notebooks\02_TrackB_Mel_CNN_FULL_CODE.txt

총 cell 수: 293
총 code cell 수: 293


In [37]:
# ============================================================
# Robust-A
# Waveform-level Low-Level Additive Noise
# ============================================================

import random
import torch
from torch.utils.data import Dataset


ROBUST_SEED = 777

random.seed(ROBUST_SEED)
np.random.seed(ROBUST_SEED)
torch.manual_seed(ROBUST_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(ROBUST_SEED)


class ICBHIRobustNoiseDataset(Dataset):

    def __init__(
        self,
        dataframe,
        training=False,
        noise_prob=0.5,
        snr_db_min=20.0,
        snr_db_max=35.0
    ):

        self.df = dataframe.reset_index(
            drop=True
        ).copy()

        self.training = training
        self.noise_prob = noise_prob

        self.snr_db_min = snr_db_min
        self.snr_db_max = snr_db_max


    def __len__(self):

        return len(self.df)


    def add_noise_snr(
        self,
        waveform
    ):

        # ----------------------------------------------------
        # waveform RMS
        # ----------------------------------------------------

        signal_rms = torch.sqrt(
            torch.mean(
                waveform ** 2
            ) + 1e-12
        )

        # 무음에 가까운 경우 augmentation 생략
        if signal_rms < 1e-7:
            return waveform

        # ----------------------------------------------------
        # random target SNR
        # ----------------------------------------------------

        target_snr_db = random.uniform(
            self.snr_db_min,
            self.snr_db_max
        )

        noise = torch.randn_like(
            waveform
        )

        noise_rms = torch.sqrt(
            torch.mean(
                noise ** 2
            ) + 1e-12
        )

        # 원하는 SNR에 맞춘 noise RMS
        target_noise_rms = (
            signal_rms
            / (
                10.0 **
                (target_snr_db / 20.0)
            )
        )

        noise = (
            noise
            / noise_rms
            * target_noise_rms
        )

        augmented = waveform + noise

        # clipping 방지
        augmented = torch.clamp(
            augmented,
            -1.0,
            1.0
        )

        return augmented


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        wav_path = row[
            "processed_file"
        ]

        waveform, sr = (
            torchaudio.load(
                wav_path
            )
        )

        # ----------------------------------------------------
        # SR check
        # ----------------------------------------------------

        if sr != SAMPLE_RATE:

            raise ValueError(
                f"Unexpected sample rate: "
                f"{sr} | {wav_path}"
            )

        # ----------------------------------------------------
        # stereo -> mono
        # ----------------------------------------------------

        if waveform.shape[0] > 1:

            waveform = waveform.mean(
                dim=0,
                keepdim=True
            )

        # ----------------------------------------------------
        # TRAIN ONLY augmentation
        # ----------------------------------------------------

        if (
            self.training
            and
            random.random()
            < self.noise_prob
        ):

            waveform = (
                self.add_noise_snr(
                    waveform
                )
            )

        # ----------------------------------------------------
        # 기존 Base와 동일한 Mel pipeline
        # ----------------------------------------------------

        mel = mel_transform(
            waveform
        )

        mel_db = (
            10.0
            * torch.log10(
                torch.clamp(
                    mel,
                    min=1e-10
                )
            )
        )

        # 기존 Base와 동일
        mel_db = (
            mel_db
            - mel_db.max()
        )

        mel_db = torch.clamp(
            mel_db,
            min=-80.0,
            max=0.0
        )

        mel_norm = (
            mel_db + 80.0
        ) / 80.0

        label = CLASS_TO_IDX[
            row["label"]
        ]

        return (
            mel_norm.float(),
            torch.tensor(
                label,
                dtype=torch.long
            )
        )


print(
    "ICBHIRobustNoiseDataset ready."
)

ICBHIRobustNoiseDataset ready.


In [40]:
# ============================================================
# Robust-A COMPLETE BOOTSTRAP + SANITY CHECK
# 현재 커널 상태와 무관하게 이 셀 하나로 실행
# ============================================================

from pathlib import Path
import random

import numpy as np
import pandas as pd

import torch
import torchaudio

from torch.utils.data import Dataset


# ============================================================
# 1. 기본 경로 / split
# ============================================================

ROOT = Path(r"D:\2026\final_project")

SPLIT_PATH = (
    ROOT
    / "data_processed"
    / "metadata"
    / "ICBHI_patient_level_split_seed777_v1.csv"
)

df = pd.read_csv(SPLIT_PATH)

train_df = df[
    df["split"].astype(str).str.lower() == "train"
].copy().reset_index(drop=True)

val_df = df[
    df["split"].astype(str).str.lower() == "validation"
].copy().reset_index(drop=True)

test_df = df[
    df["split"].astype(str).str.lower() == "test"
].copy().reset_index(drop=True)


# ============================================================
# 2. 원본 Base 설정
# ============================================================

SAMPLE_RATE = 4000
TARGET_DURATION = 4.0
TARGET_SAMPLES = 16000

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 128
FMIN = 0
FMAX = 2000

CLASS_NAMES = [
    "Both",
    "Crackle",
    "Normal",
    "Wheeze"
]

CLASS_TO_IDX = {
    name: idx
    for idx, name in enumerate(CLASS_NAMES)
}

IDX_TO_CLASS = {
    idx: name
    for name, idx in CLASS_TO_IDX.items()
}


# ============================================================
# 3. Mel transform
# ============================================================

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=FMIN,
    f_max=FMAX,
    power=2.0
)


# ============================================================
# 4. Base preprocessing 함수
#    원본 ICBHIMelDataset과 동일
# ============================================================

def load_base_mel(row):

    wav_path = row["processed_file"]

    waveform, sr = torchaudio.load(
        wav_path
    )

    if sr != SAMPLE_RATE:
        raise ValueError(
            f"Unexpected sample rate: {sr} | {wav_path}"
        )

    if waveform.shape[0] > 1:
        waveform = waveform.mean(
            dim=0,
            keepdim=True
        )

    mel = mel_transform(
        waveform
    )

    mel_db = 10.0 * torch.log10(
        torch.clamp(
            mel,
            min=1e-10
        )
    )

    # sample별 peak = 0 dB
    mel_db = mel_db - mel_db.max()

    mel_db = torch.clamp(
        mel_db,
        min=-80.0,
        max=0.0
    )

    mel_norm = (
        mel_db + 80.0
    ) / 80.0

    label = CLASS_TO_IDX[
        row["label"]
    ]

    return (
        mel_norm.float(),
        torch.tensor(
            label,
            dtype=torch.long
        )
    )


# ============================================================
# 5. Robust-A Dataset
# ============================================================

class ICBHIRobustNoiseDataset(Dataset):

    def __init__(
        self,
        dataframe,
        training=False,
        noise_prob=0.5,
        snr_db_min=20.0,
        snr_db_max=35.0
    ):

        self.df = dataframe.reset_index(
            drop=True
        ).copy()

        self.training = training
        self.noise_prob = noise_prob
        self.snr_db_min = snr_db_min
        self.snr_db_max = snr_db_max


    def __len__(self):

        return len(self.df)


    def add_noise_snr(
        self,
        waveform
    ):

        signal_rms = torch.sqrt(
            torch.mean(
                waveform ** 2
            ) + 1e-12
        )

        if signal_rms < 1e-7:
            return waveform

        target_snr_db = random.uniform(
            self.snr_db_min,
            self.snr_db_max
        )

        noise = torch.randn_like(
            waveform
        )

        noise_rms = torch.sqrt(
            torch.mean(
                noise ** 2
            ) + 1e-12
        )

        target_noise_rms = (
            signal_rms
            /
            (
                10.0 **
                (target_snr_db / 20.0)
            )
        )

        noise = (
            noise
            / noise_rms
            * target_noise_rms
        )

        augmented = waveform + noise

        augmented = torch.clamp(
            augmented,
            -1.0,
            1.0
        )

        return augmented


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        waveform, sr = torchaudio.load(
            row["processed_file"]
        )

        if sr != SAMPLE_RATE:
            raise ValueError(
                f"Unexpected sample rate: {sr}"
            )

        if waveform.shape[0] > 1:
            waveform = waveform.mean(
                dim=0,
                keepdim=True
            )

        # Train에서만 augmentation
        if (
            self.training
            and
            random.random() < self.noise_prob
        ):
            waveform = self.add_noise_snr(
                waveform
            )

        mel = mel_transform(
            waveform
        )

        mel_db = 10.0 * torch.log10(
            torch.clamp(
                mel,
                min=1e-10
            )
        )

        mel_db = mel_db - mel_db.max()

        mel_db = torch.clamp(
            mel_db,
            min=-80.0,
            max=0.0
        )

        mel_norm = (
            mel_db + 80.0
        ) / 80.0

        label = CLASS_TO_IDX[
            row["label"]
        ]

        return (
            mel_norm.float(),
            torch.tensor(
                label,
                dtype=torch.long
            )
        )


# ============================================================
# 6. Dataset 생성
# ============================================================

robust_train_dataset = ICBHIRobustNoiseDataset(
    train_df,
    training=True,

    # sanity check이므로 100% 적용
    noise_prob=1.0,

    snr_db_min=20.0,
    snr_db_max=35.0
)

robust_val_dataset = ICBHIRobustNoiseDataset(
    val_df,
    training=False,
    noise_prob=0.0
)


# ============================================================
# 7. Train sanity check
# ============================================================

check_indices = [0, 1, 2, 10, 25]

rows = []

for idx in check_indices:

    row = train_df.iloc[idx]

    # 같은 random state
    random.seed(777 + idx)
    torch.manual_seed(777 + idx)

    base_x, base_y = load_base_mel(
        row
    )

    random.seed(777 + idx)
    torch.manual_seed(777 + idx)

    robust_x, robust_y = (
        robust_train_dataset[idx]
    )

    diff = torch.abs(
        base_x - robust_x
    )

    rows.append({

        "idx": idx,

        "label":
            IDX_TO_CLASS[
                int(base_y)
            ],

        "base_shape":
            tuple(base_x.shape),

        "robust_shape":
            tuple(robust_x.shape),

        "base_min":
            float(base_x.min()),

        "base_max":
            float(base_x.max()),

        "robust_min":
            float(robust_x.min()),

        "robust_max":
            float(robust_x.max()),

        "mean_abs_diff":
            float(diff.mean()),

        "max_abs_diff":
            float(diff.max()),

        "same_label":
            base_y.item()
            == robust_y.item(),

        "same_shape":
            base_x.shape
            == robust_x.shape,

        "nan_base":
            bool(
                torch.isnan(base_x).any()
            ),

        "nan_robust":
            bool(
                torch.isnan(robust_x).any()
            ),
    })


sanity_df = pd.DataFrame(
    rows
)

print("=" * 90)
print("ROBUST-A COMPLETE SANITY CHECK")
print("=" * 90)

print("\nSplit:")
print(
    "Train =", len(train_df),
    "| Validation =", len(val_df),
    "| Test =", len(test_df)
)

print("\nClass mapping:")
print(CLASS_TO_IDX)

print("\nTrain 비교:")
display(sanity_df)


# ============================================================
# 8. 자동 판정
# ============================================================

print("\n[자동 판정]")

print(
    "Label 동일       :",
    sanity_df["same_label"].all()
)

print(
    "Shape 동일       :",
    sanity_df["same_shape"].all()
)

print(
    "NaN 없음         :",
    (
        ~sanity_df["nan_base"]
        &
        ~sanity_df["nan_robust"]
    ).all()
)

print(
    "Base 0~1         :",
    (
        (sanity_df["base_min"] >= 0)
        &
        (sanity_df["base_max"] <= 1)
    ).all()
)

print(
    "Robust 0~1       :",
    (
        (sanity_df["robust_min"] >= 0)
        &
        (sanity_df["robust_max"] <= 1)
    ).all()
)

print(
    "Noise 변화 발생 :",
    (
        sanity_df["mean_abs_diff"] > 0
    ).all()
)


# ============================================================
# 9. Validation 동일성
# ============================================================

print("\n" + "=" * 90)
print("VALIDATION 동일성 확인")
print("=" * 90)

for idx in [0, 1, 2]:

    base_x, base_y = load_base_mel(
        val_df.iloc[idx]
    )

    robust_x, robust_y = (
        robust_val_dataset[idx]
    )

    max_diff = torch.abs(
        base_x - robust_x
    ).max().item()

    print(
        f"idx={idx:3d} | "
        f"label={IDX_TO_CLASS[int(base_y)]} | "
        f"max_diff={max_diff:.10f}"
    )

ROBUST-A COMPLETE SANITY CHECK

Split:
Train = 4737 | Validation = 1141 | Test = 1018

Class mapping:
{'Both': 0, 'Crackle': 1, 'Normal': 2, 'Wheeze': 3}

Train 비교:


,idx,label,base_shape,robust_shape,base_min,base_max,robust_min,robust_max,mean_abs_diff,max_abs_diff,same_label,same_shape,nan_base,nan_robust
0,0,Normal,"(1, 128, 126)","(1, 128, 126)",0.0,1.0,0.005862,1.0,0.254926,0.417152,True,True,False,False
1,1,Normal,"(1, 128, 126)","(1, 128, 126)",0.0,1.0,0.051434,1.0,0.296380,0.530139,True,True,False,False
2,2,Normal,"(1, 128, 126)","(1, 128, 126)",0.0,1.0,0.000000,1.0,0.200805,0.397335,True,True,False,False
3,10,Normal,"(1, 128, 126)","(1, 128, 126)",0.0,1.0,0.000000,1.0,0.220232,0.455620,True,True,False,False
4,25,Wheeze,"(1, 128, 126)","(1, 128, 126)",0.0,1.0,0.048516,1.0,0.217717,0.443958,True,True,False,False



[자동 판정]
Label 동일       : True
Shape 동일       : True
NaN 없음         : True
Base 0~1         : True
Robust 0~1       : True
Noise 변화 발생 : True

VALIDATION 동일성 확인
idx=  0 | label=Normal | max_diff=0.0000000000
idx=  1 | label=Normal | max_diff=0.0000000000
idx=  2 | label=Normal | max_diff=0.0000000000


In [ ]:
# ============================================================
# Robust-A TRAINING
# Low-level additive noise
# ============================================================

from pathlib import Path
import random
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import models

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)


# ============================================================
# 1. Reproducibility
# ============================================================

SEED = 777

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 90)
print("ROBUST-A TRAINING")
print("=" * 90)
print("Device:", device)


# ============================================================
# 2. Robust Train / Base Validation Dataset
# ============================================================

robust_train_dataset = ICBHIRobustNoiseDataset(
    train_df,
    training=True,
    noise_prob=0.5,
    snr_db_min=20.0,
    snr_db_max=35.0
)

robust_val_dataset = ICBHIRobustNoiseDataset(
    val_df,
    training=False,
    noise_prob=0.0
)


# ============================================================
# 3. DataLoader
# ============================================================

BATCH_SIZE = 32

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

robust_train_loader = DataLoader(
    robust_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    generator=train_generator
)

robust_val_loader = DataLoader(
    robust_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(
    "Train:",
    len(robust_train_dataset),
    "| Validation:",
    len(robust_val_dataset)
)


# ============================================================
# 4. Class weights
#    Base와 동일 방식
# ============================================================

train_labels = (
    train_df["label"]
    .map(CLASS_TO_IDX)
    .values
)

classes = np.array([0, 1, 2, 3])

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=device
)

print("\nClass weights:")

for idx, weight in enumerate(class_weights_np):
    print(
        f"{CLASS_NAMES[idx]:10s}: "
        f"{weight:.6f}"
    )


# ============================================================
# 5. Loss
# ============================================================

criterion_robust = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# 6. AlexNet
#    Base와 동일
# ============================================================

robust_alexnet = models.alexnet(
    weights=models.AlexNet_Weights.IMAGENET1K_V1
)

robust_alexnet.classifier[6] = nn.Linear(
    robust_alexnet.classifier[6].in_features,
    4
)

robust_alexnet = robust_alexnet.to(
    device
)


# ============================================================
# 7. Optimizer
# ============================================================

robust_optimizer = torch.optim.Adam(
    robust_alexnet.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ============================================================
# 8. AlexNet input
# ============================================================

def prepare_robust_alexnet_input(x):

    x = F.interpolate(
        x,
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    )

    x = x.repeat(
        1, 3, 1, 1
    )

    return x


# ============================================================
# 9. Evaluation
# ============================================================

def evaluate_robust(
    model,
    loader
):

    model.eval()

    all_preds = []
    all_labels = []

    running_loss = 0.0

    with torch.no_grad():

        for x, y in loader:

            x = prepare_robust_alexnet_input(
                x
            ).to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            )

            logits = model(x)

            loss = criterion_robust(
                logits,
                y
            )

            running_loss += (
                loss.item()
                * x.size(0)
            )

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                y.cpu().numpy()
            )

    avg_loss = (
        running_loss
        / len(loader.dataset)
    )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    balanced_acc = balanced_accuracy_score(
        all_labels,
        all_preds
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "macro_f1": macro_f1
    }


# ============================================================
# 10. Training loop
# ============================================================

MAX_EPOCHS = 40
PATIENCE = 10

best_state = None
best_macro_f1 = -1.0
best_epoch = 0

patience_counter = 0
history = []

for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    robust_alexnet.train()

    running_loss = 0.0

    for x, y in robust_train_loader:

        x = prepare_robust_alexnet_input(
            x
        ).to(
            device,
            non_blocking=True
        )

        y = y.to(
            device,
            non_blocking=True
        )

        robust_optimizer.zero_grad()

        logits = robust_alexnet(
            x
        )

        loss = criterion_robust(
            logits,
            y
        )

        loss.backward()

        robust_optimizer.step()

        running_loss += (
            loss.item()
            * x.size(0)
        )

    train_loss = (
        running_loss
        / len(
            robust_train_loader.dataset
        )
    )

    val_metrics = evaluate_robust(
        robust_alexnet,
        robust_val_loader
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss":
            val_metrics["loss"],
        "val_accuracy":
            val_metrics["accuracy"],
        "val_balanced_accuracy":
            val_metrics[
                "balanced_accuracy"
            ],
        "val_macro_f1":
            val_metrics["macro_f1"]
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_metrics['loss']:.4f} | "
        f"Val Acc {val_metrics['accuracy']:.4f} | "
        f"Bal Acc {val_metrics['balanced_accuracy']:.4f} | "
        f"Macro F1 {val_metrics['macro_f1']:.4f}"
    )

    # Validation Macro F1 기준
    if (
        val_metrics["macro_f1"]
        > best_macro_f1
    ):

        best_macro_f1 = (
            val_metrics["macro_f1"]
        )

        best_epoch = epoch

        best_state = copy.deepcopy(
            robust_alexnet.state_dict()
        )

        patience_counter = 0

    else:

        patience_counter += 1

    if patience_counter >= PATIENCE:

        print(
            "\nEarly stopping:",
            f"no Macro F1 improvement "
            f"for {PATIENCE} epochs."
        )

        break


# ============================================================
# 11. Best model restore
# ============================================================

robust_alexnet.load_state_dict(
    best_state
)

robust_history = pd.DataFrame(
    history
)


print("\n" + "=" * 90)
print("ROBUST-A TRAINING COMPLETE")
print("=" * 90)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best Validation Macro F1:",
    round(
        best_macro_f1,
        6
    )
)


# ============================================================
# 12. 저장
# ============================================================

OUT_DIR = (
    ROOT
    / "results"
    / "robustness"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH = (
    OUT_DIR
    / "AlexNet_RobustA_Noise_seed777.pth"
)

HISTORY_PATH = (
    OUT_DIR
    / "AlexNet_RobustA_Noise_history_seed777.csv"
)

torch.save(
    robust_alexnet.state_dict(),
    MODEL_PATH
)

robust_history.to_csv(
    HISTORY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved model:")
print(MODEL_PATH)

print("\nSaved history:")
print(HISTORY_PATH)

ROBUST-A TRAINING
Device: cpu
Train: 4737 | Validation: 1141

Class weights:
Both      : 3.393266
Crackle   : 0.941375
Normal    : 0.465507
Wheeze    : 2.020904


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to C:\Users\정용호/.cache\torch\hub\checkpoints\alexnet-owt-7be5be79.pth
100%|██████████| 233M/233M [00:11<00:00, 20.7MB/s] 


Epoch 01 | Train Loss 1.3594 | Val Loss 1.4638 | Val Acc 0.3059 | Bal Acc 0.2498 | Macro F1 0.2080
Epoch 02 | Train Loss 1.2100 | Val Loss 1.2708 | Val Acc 0.3690 | Bal Acc 0.3673 | Macro F1 0.3101
Epoch 03 | Train Loss 1.1554 | Val Loss 1.3182 | Val Acc 0.3620 | Bal Acc 0.3732 | Macro F1 0.3219
Epoch 04 | Train Loss 1.0683 | Val Loss 1.3122 | Val Acc 0.4671 | Bal Acc 0.3415 | Macro F1 0.3267
Epoch 05 | Train Loss 1.0332 | Val Loss 1.2418 | Val Acc 0.4759 | Bal Acc 0.3822 | Macro F1 0.3761
Epoch 06 | Train Loss 0.9600 | Val Loss 1.2950 | Val Acc 0.4873 | Bal Acc 0.3982 | Macro F1 0.3849
Epoch 07 | Train Loss 0.9252 | Val Loss 1.3471 | Val Acc 0.4023 | Bal Acc 0.3720 | Macro F1 0.3251
Epoch 08 | Train Loss 0.8960 | Val Loss 1.3547 | Val Acc 0.4917 | Bal Acc 0.3696 | Macro F1 0.3526
Epoch 09 | Train Loss 0.8584 | Val Loss 1.4124 | Val Acc 0.4680 | Bal Acc 0.3612 | Macro F1 0.3388
Epoch 10 | Train Loss 0.7993 | Val Loss 1.4826 | Val Acc 0.3839 | Bal Acc 0.3640 | Macro F1 0.3276
Epoch 11 |